In [1]:
# ===================== BOOTSTRAP OFFLINE (RUN ALL - KAGGLE) =====================
import os
import sys
import glob
import tempfile
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()


def _pick_first_existing(candidates: list[str]) -> str | None:
    """Retorna o primeiro caminho existente da lista (ou None)."""
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate
    return None


def _discover_wheels_archive() -> str | None:
    """Descobre automaticamente o archive de wheels no Kaggle/input."""
    explicit = os.getenv('AIMO3_WHEELS_ARCHIVE')
    if explicit and os.path.exists(explicit):
        return explicit

    patterns = [
        '/kaggle/input/*/wheels.tar.gz',
        '/kaggle/input/*/*/wheels.tar.gz',
    ]

    for pattern in patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return matches[0]

    return None


OFFLINE_PATHS = {}
OFFLINE_PATHS['setup_dir'] = os.getenv(
    'AIMO3_SETUP_DIR',
    '/kaggle/tmp/setup' if IS_KAGGLE else os.path.join(tempfile.gettempdir(), 'aimo3_setup'),
)

OFFLINE_PATHS['wheels_archive'] = _discover_wheels_archive()
OFFLINE_PATHS['wheels_dir'] = os.getenv('AIMO3_WHEELS_DIR') or (
    '/kaggle/input/aimo-3-utils/wheels' if IS_KAGGLE else None
)

OFFLINE_PATHS['model_path'] = _pick_first_existing([
    os.getenv('AIMO3_MODEL_PATH', ''),
    '/kaggle/input/gpt-oss-120b/transformers/default/1',
    '/kaggle/input/gpt-oss-20b/transformers/default/1',
    '/kaggle/working/models/gpt-oss-120b',
])

OFFLINE_PATHS['reference_csv'] = _pick_first_existing([
    os.getenv('AIMO3_REFERENCE_CSV', ''),
    '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv',
    '/kaggle/working/reference.csv',
])

# Sinalizadores offline para evitar acesso à internet no Run All.
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('PIP_NO_INDEX', '1')
os.environ.setdefault('PIP_DISABLE_PIP_VERSION_CHECK', '1')

if OFFLINE_PATHS['model_path'] is not None:
    os.environ['AIMO3_MODEL_PATH'] = OFFLINE_PATHS['model_path']
if OFFLINE_PATHS['reference_csv'] is not None:
    os.environ['AIMO3_REFERENCE_CSV'] = OFFLINE_PATHS['reference_csv']

print('[BOOTSTRAP] Offline bootstrap inicializado')
print(f"[BOOTSTRAP] IS_KAGGLE={IS_KAGGLE}")
print(f"[BOOTSTRAP] setup_dir={OFFLINE_PATHS['setup_dir']}")
print(f"[BOOTSTRAP] wheels_archive={OFFLINE_PATHS['wheels_archive']}")
print(f"[BOOTSTRAP] wheels_dir_hint={OFFLINE_PATHS['wheels_dir']}")
print(f"[BOOTSTRAP] model_path={OFFLINE_PATHS['model_path']}")
print(f"[BOOTSTRAP] reference_csv={OFFLINE_PATHS['reference_csv']}")

[BOOTSTRAP] Offline bootstrap inicializado
[BOOTSTRAP] IS_KAGGLE=True
[BOOTSTRAP] setup_dir=/kaggle/tmp/setup
[BOOTSTRAP] wheels_archive=/kaggle/input/aimo-3-utils/wheels.tar.gz
[BOOTSTRAP] wheels_dir_hint=/kaggle/input/aimo-3-utils/wheels
[BOOTSTRAP] model_path=/kaggle/input/gpt-oss-120b/transformers/default/1
[BOOTSTRAP] reference_csv=/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv


In [2]:
def set_env(
    input_archive: str | None,
    temp_dir: str,
    wheels_dir: str | None = None,
    packages: list[str] | None = None,
    requirements_file: str | None = None,
    force_reinstall: bool = False,
    required_modules: dict[str, str] | None = None,
) -> dict:
    """Prepara ambiente offline idempotente para execução com Run All.

    Esta função:
    1) resolve/extrai wheelhouse local,
    2) instala dependências sem internet,
    3) valida imports críticos logo após o pip,
    4) retorna metadados de setup para auditoria.
    """
    import importlib
    import importlib.util
    import json
    import site
    import subprocess
    import tarfile

    default_packages = [
        'unsloth',
        'trl',
        'vllm',
        'openai_harmony',
        'datasets',
        'polars',
        'openai',
    ]
    packages = packages or default_packages
    required_modules = required_modules or {'openai_harmony': 'openai_harmony'}

    os.makedirs(temp_dir, exist_ok=True)
    marker_path = os.path.join(temp_dir, '.offline_install_ok.json')

    extracted_now = False
    fallback_wheelhouse = os.path.join(temp_dir, 'wheels')

    def _refresh_site_packages() -> None:
        """Atualiza sys.path após instalações via pip no runtime do notebook."""
        importlib.invalidate_caches()

        site_dirs = []
        try:
            site_dirs.extend(site.getsitepackages())
        except Exception:
            pass

        try:
            user_site = site.getusersitepackages()
            if isinstance(user_site, str):
                site_dirs.append(user_site)
            else:
                site_dirs.extend(list(user_site))
        except Exception:
            pass

        for site_dir in site_dirs:
            if site_dir and os.path.isdir(site_dir) and site_dir not in sys.path:
                sys.path.append(site_dir)

    def _list_wheels(directory: str | None) -> list[str]:
        if not directory or not os.path.isdir(directory):
            return []
        return sorted(glob.glob(os.path.join(directory, '*.whl')))

    def _find_matching_wheels(package_name: str, directory: str | None) -> list[str]:
        normalized_names = {
            package_name.lower(),
            package_name.replace('-', '_').lower(),
            package_name.replace('_', '-').lower(),
        }
        matches = []
        for wheel_path in _list_wheels(directory):
            wheel_name = os.path.basename(wheel_path).lower()
            if any(wheel_name.startswith(f'{name}-') for name in normalized_names):
                matches.append(wheel_path)
        return matches

    hint_wheels = _list_wheels(wheels_dir)
    extracted_wheels = _list_wheels(fallback_wheelhouse)

    if hint_wheels:
        wheelhouse = wheels_dir
        wheelhouse_source = 'attached_wheels_dir'
    else:
        wheelhouse = fallback_wheelhouse
        wheelhouse_source = 'extracted_archive' if extracted_wheels else 'missing'

        if input_archive and os.path.exists(input_archive):
            print(f'[SET_ENV] Extraindo wheelhouse: {input_archive}')
            with tarfile.open(input_archive, 'r:gz') as archive:
                archive.extractall(path=temp_dir)
            extracted_now = True
            extracted_wheels = _list_wheels(fallback_wheelhouse)

            if extracted_wheels:
                wheelhouse_source = 'extracted_archive'
            elif hint_wheels:
                wheelhouse = wheels_dir
                wheelhouse_source = 'attached_wheels_dir'
        elif extracted_wheels:
            wheelhouse_source = 'extracted_archive'

    wheel_files = _list_wheels(wheelhouse)
    if not wheel_files:
        raise RuntimeError(
            '[SET_ENV] Nenhum .whl encontrado em um wheelhouse válido. '
            f'wheels_dir={wheels_dir} (wheels={len(hint_wheels)}) | '
            f'archive={input_archive} | extracted_dir={fallback_wheelhouse} (wheels={len(extracted_wheels)})'
        )

    print(
        f'[SET_ENV] wheelhouse selecionado: {wheelhouse} '
        f'(source={wheelhouse_source}, wheels={len(wheel_files)})'
    )

    requirements_file = requirements_file or os.getenv('AIMO3_OFFLINE_REQUIREMENTS')
    req_exists = bool(requirements_file and os.path.exists(requirements_file))

    install_signature = {
        'wheelhouse': wheelhouse,
        'packages': sorted(packages),
        'requirements_file': requirements_file if req_exists else None,
        'required_modules': sorted(required_modules.items()),
    }

    needs_install = force_reinstall
    if not needs_install and os.path.exists(marker_path):
        try:
            with open(marker_path, 'r', encoding='utf-8') as file_object:
                old_signature = json.load(file_object)
            needs_install = old_signature != install_signature
        except Exception:
            needs_install = True
    else:
        needs_install = True

    if needs_install:
        cmd = [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--no-index',
            '--find-links',
            wheelhouse,
            '--disable-pip-version-check',
        ]

        if req_exists:
            cmd.extend(['-r', requirements_file])

        cmd.extend(packages)
        print(f'[SET_ENV] Instalando pacotes offline ({len(packages)} pacotes-alvo)...')
        subprocess.run(cmd, check=True)

        with open(marker_path, 'w', encoding='utf-8') as file_object:
            json.dump(install_signature, file_object, ensure_ascii=False, indent=2)
        print('[SET_ENV] Instalação concluída e marker atualizado.')
    else:
        print('[SET_ENV] Instalação já validada anteriormente; pulando pip install.')

    def _ensure_importable(module_name: str, package_name: str) -> None:
        """Garante que um módulo crítico esteja importável no kernel atual."""
        _refresh_site_packages()
        if importlib.util.find_spec(module_name) is not None:
            return

        matching_wheels = _find_matching_wheels(package_name, wheelhouse)
        if matching_wheels:
            print(
                f'[SET_ENV][WARN] {module_name} ausente após setup; forçando reinstalação offline de {package_name}.'
            )
            reinstall_cmd = [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--no-index',
                '--find-links',
                wheelhouse,
                '--force-reinstall',
                '--disable-pip-version-check',
                package_name,
            ]
            subprocess.run(reinstall_cmd, check=True)
            _refresh_site_packages()

        if importlib.util.find_spec(module_name) is None:
            matching_wheels = _find_matching_wheels(package_name, wheelhouse)
            raise RuntimeError(
                f'[SET_ENV] Módulo obrigatório não importável: {module_name}. '
                f'wheelhouse={wheelhouse} | wheels_detectados={len(matching_wheels)} | '
                f'package={package_name}. Verifique se o dataset offline contém o wheel correto.'
            )

    _refresh_site_packages()
    validated_modules = []
    for module_name, package_name in required_modules.items():
        _ensure_importable(module_name, package_name)
        validated_modules.append(module_name)

    tiktoken_candidates = [
        os.path.join(temp_dir, 'tiktoken_encodings'),
        os.path.join(os.path.dirname(wheelhouse), 'tiktoken_encodings'),
    ]
    if input_archive:
        tiktoken_candidates.append(
            os.path.join(os.path.dirname(input_archive), 'tiktoken_encodings')
        )

    tiktoken_dir = _pick_first_existing(tiktoken_candidates)
    if tiktoken_dir:
        os.environ['TIKTOKEN_ENCODINGS_BASE'] = tiktoken_dir
    else:
        print('[SET_ENV][WARN] tiktoken_encodings não encontrado; mantendo valor atual de TIKTOKEN_ENCODINGS_BASE.')

    return {
        'temp_dir': temp_dir,
        'wheelhouse': wheelhouse,
        'wheelhouse_source': wheelhouse_source,
        'wheel_count': len(wheel_files),
        'extracted_now': extracted_now,
        'tiktoken_encodings_dir': tiktoken_dir,
        'marker_path': marker_path,
        'validated_modules': validated_modules,
    }

In [3]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz',
    temp_dir='/kaggle/tmp/setup',
    packages=[
        'vllm',
        'openai_harmony',
        'openai',
        'polars',
        'datasets',
        'transformers',
        'trl',
        'peft',
        'accelerate',
        # unsloth mantido opcional para comparações locais;
        # fluxo principal de LoRA neste notebook usa transformers+peft.
    ],
    required_modules={
        'openai_harmony': 'openai_harmony',
        'transformers': 'transformers',
        'trl': 'trl',
        'peft': 'peft',
    },
)

[SET_ENV] Extraindo wheelhouse: /kaggle/input/aimo-3-utils/wheels.tar.gz


/tmp/ipykernel_106/3338888467.py:97: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(path=temp_dir)


[SET_ENV] wheelhouse selecionado: /kaggle/tmp/setup/wheels (source=extracted_archive, wheels=178)
[SET_ENV] Instalando pacotes offline (9 pacotes-alvo)...
Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/lm_format_enforcer-0.11.3-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/llguidance-1.3.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/outlines_core-0.2.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/diskcache-5.6.3-py3-none-any.whl (from vllm)
Processing /kaggl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.2.19 which is incompatible.
cudf-cu12 25.6.0 requires cuda-python<13.0a0,>=12.6.2, but you have cuda-python 13.1.1 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
cuml-cu12 25.6.0 requires cuda-python<13.0a0,>=12.6.2, but you have cuda-python 13.1.1 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pylibraft-cu12 25.6.0 requires cuda-python<13.0a0,>=12.6.2, but you have cuda-python 13.1

{'temp_dir': '/kaggle/tmp/setup',
 'wheelhouse': '/kaggle/tmp/setup/wheels',
 'wheelhouse_source': 'extracted_archive',
 'wheel_count': 178,
 'extracted_now': True,
 'tiktoken_encodings_dir': '/kaggle/tmp/setup/tiktoken_encodings',
 'marker_path': '/kaggle/tmp/setup/.offline_install_ok.json',
 'validated_modules': ['openai_harmony', 'transformers', 'trl', 'peft']}

In [4]:
# ===================== OPTIONAL: UNSLOTH INSTALL (OFF BY DEFAULT) =====================
install_unsloth = os.getenv('AIMO3_INSTALL_UNSLOTH', '0') == '1'

if install_unsloth:
    import subprocess
    wheelhouse = '/kaggle/tmp/setup/wheels'
    print('[SET_ENV][OPTIONAL] Instalando unsloth sob demanda...')
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install',
            '--no-index',
            '--find-links', wheelhouse,
            '--disable-pip-version-check',
            'unsloth',
        ],
        check=True,
    )
    print('[SET_ENV][OPTIONAL] unsloth instalado.')
else:
    print('[SET_ENV][OPTIONAL] unsloth desativado (AIMO3_INSTALL_UNSLOTH=0).')

[SET_ENV][OPTIONAL] unsloth desativado (AIMO3_INSTALL_UNSLOTH=0).


In [5]:
import subprocess
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [6]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

# Estabilidade para execução longa + logs limpos em Kaggle
os.environ.setdefault('WANDB_DISABLED', 'true')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('WANDB_CONSOLE', 'off')
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')
os.environ.setdefault('PYTHONWARNINGS', 'ignore')

'ignore'

In [7]:
class CFG:
    """Centraliza prompts e parâmetros operacionais do pipeline AIMO3.

    A classe funciona como um namespace imutável para reunir os textos-base
    usados pelo modelo e os hiperparâmetros que governam inferência, tempo,
    paralelismo e execução do servidor local.
    """

    # Prompt principal usado para guiar o raciocínio matemático do modelo.
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'

        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'

        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'

        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'

        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'

        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )

    # Instruções específicas do tool de Python usado durante a inferência.
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'

        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'

        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )

    # Preferências adicionais para incentivar cálculo simbólico e validação numérica.
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'

        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'

        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'

        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'

        '# Modular Arithmetic Toolkit:\n'
        '- Use `pow(base, exp, mod)` for fast modular exponentiation\n'
        '- Use Euler phi / multiplicative order only after proving gcd(a, n)=1\n'
        '- Use CRT decomposition for composite moduli and recombine carefully\n'
        '- For gcd(a,n) != 1, avoid naive exponent reduction and use valuation/divisibility checks\n\n'

        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )

    # Identidade e caminhos do modelo servido localmente.
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'

    # Configurações de dtype e cache do vLLM.
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    # Limites de tempo por problema e por notebook.
    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    # Limites de timeout para sessão, Jupyter e sandbox.
    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    # Hiperparâmetros de geração e paralelismo.
    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    # Parâmetros de execução no GPU e sampling.
    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

    # Estratégia pedida: perfil mais próximo do notebook 47-50 + foco modular.
    compat_mode_47 = os.getenv('AIMO3_COMPAT_MODE_47', '1') == '1'
    modular_focus_mode = os.getenv('AIMO3_MODULAR_FOCUS_MODE', '1') == '1'
    enable_modular_guidance = os.getenv('AIMO3_ENABLE_MODULAR_GUIDANCE', '1') == '1'
    max_modular_guidance_blocks = 3

    # Fontes estudadas pelo usuário (usadas para curadoria manual da KB modular).
    modular_source_documents = [
        'D:/dowloads/mit6_1200j_s24_lec09.pdf',
        'D:/dowloads/DiscMathLecture08.pdf',
        'D:/dowloads/mit6_042js15_session13.pdf',
        'D:/dowloads/DiscMathLecture08 (1).pdf',
    ]

    # Enriquecimento generalista por problema (Wikimedia + teoremas).
    enable_wikimedia_context = os.getenv(
        'AIMO3_ENABLE_WIKIMEDIA_CONTEXT',
        '0' if compat_mode_47 else '1',
    ) == '1'
    enable_theorem_guidance = os.getenv('AIMO3_ENABLE_THEOREM_GUIDANCE', '1') == '1'
    max_context_domains = 2
    max_context_items = 2 if compat_mode_47 else 4
    max_theorem_blocks = 5 if modular_focus_mode else 4
    max_context_chars_per_item = 450
    max_problem_context_chars = 1800
    context_query_chars = 180
    always_attempt_context_lookup = not compat_mode_47
    theorem_mode = 'conditional'


In [8]:
if 'CFG' not in globals():
    raise RuntimeError(
        '[SEED] CFG não está definido. Execute as células anteriores até a célula 6 antes de continuar.'
    )

import random

def _apply_seed(seed_value: int) -> None:
    """Aplica seed de forma leve, sem depender do import global de transformers."""
    random.seed(seed_value)

    if 'np' in globals():
        np.random.seed(seed_value)

    if 'torch' in globals():
        torch.manual_seed(seed_value)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed_value)
            torch.cuda.manual_seed_all(seed_value)

    if 'set_seed' in globals():
        try:
            set_seed(seed_value)
            print('[SEED] set_seed global reutilizado com sucesso.')
            return
        except Exception as exc:
            print(f'[SEED][WARN] set_seed global falhou; usando fallback leve. Motivo: {exc}')

    print('[SEED][INFO] Aplicando fallback leve de seed (random/numpy/torch).')

_apply_seed(CFG.seed)
print(f'[SEED] Seed aplicada com sucesso: {CFG.seed}')

[SEED][INFO] Aplicando fallback leve de seed (random/numpy/torch).
[SEED] Seed aplicada com sucesso: 42


In [9]:
import os, glob

# habilita LoRA em serving
os.environ["AIMO3_LORA_ENABLED"] = "1"
os.environ["AIMO3_LORA_TRAIN_ENABLED"] = "0"  # usar adapter pronto

# ajuste para o caminho REAL do adapter no Kaggle
os.environ["AIMO3_LORA_ADAPTER_PATH"] = "/kaggle/input/SEU_DATASET_LORA/aimo3_lora"
os.environ["AIMO3_LORA_MODULE_NAME"] = "aimo3_lora"

p = os.environ["AIMO3_LORA_ADAPTER_PATH"]
print("adapter_dir existe?", os.path.isdir(p))
print("adapter_config.json existe?", os.path.exists(os.path.join(p, "adapter_config.json")))
print("adapter_model*:", glob.glob(os.path.join(p, "adapter_model*")))

adapter_dir existe? False
adapter_config.json existe? False
adapter_model*: []


In [10]:

# ===================== STRATEGY RUNTIME CONFIG (SC + RERANK + TOOL LITE + LoRA) =====================
def _env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

def _env_int(name: str, default: int) -> int:
    try:
        return int(os.getenv(name, str(default)))
    except Exception:
        return default

def _env_float(name: str, default: float) -> float:
    try:
        return float(os.getenv(name, str(default)))
    except Exception:
        return default

def _env_float_list(name: str, default_values: list[float]) -> list[float]:
    raw = os.getenv(name)
    if not raw:
        return list(default_values)
    values = []
    for token in raw.split(','):
        token = token.strip()
        if not token:
            continue
        try:
            values.append(float(token))
        except Exception:
            pass
    return values if values else list(default_values)

def _env_int_list(name: str, default_values: list[int]) -> list[int]:
    raw = os.getenv(name)
    if not raw:
        return list(default_values)
    values = []
    for token in raw.split(','):
        token = token.strip()
        if not token:
            continue
        try:
            values.append(int(token))
        except Exception:
            pass
    return values if values else list(default_values)

# Ajustes de inferência orientados a AIMO3 (quick wins antes de treino pesado)
setattr(CFG, 'attempts', _env_int('AIMO3_ATTEMPTS', max(getattr(CFG, 'attempts', 8), 10)))
setattr(CFG, 'early_stop', _env_int('AIMO3_EARLY_STOP', max(getattr(CFG, 'early_stop', 4), 7)))
setattr(CFG, 'temperature', _env_float('AIMO3_TEMPERATURE', 0.9))
setattr(CFG, 'min_p', _env_float('AIMO3_MIN_P', 0.03))
setattr(CFG, 'top_logprobs', _env_int('AIMO3_TOP_LOGPROBS', max(getattr(CFG, 'top_logprobs', 5), 8)))

# Self-consistency adaptativo (diversidade controlada)
setattr(CFG, 'use_adaptive_sampling', _env_bool('AIMO3_USE_ADAPTIVE_SAMPLING', True))
setattr(CFG, 'adaptive_temperatures', _env_float_list(
    'AIMO3_ADAPTIVE_TEMPERATURES',
    [0.55, 0.60, 0.70, 0.80, 0.90, 0.95],
))
setattr(CFG, 'adaptive_min_p', _env_float_list(
    'AIMO3_ADAPTIVE_MIN_P',
    [0.02, 0.02, 0.025, 0.03, 0.035, 0.04],
))
setattr(CFG, 'adaptive_turns', _env_int_list(
    'AIMO3_ADAPTIVE_TURNS',
    [128, 120, 112, 96, 80, 64],
))

# Reflexion e rerank sob baixa confiança
setattr(CFG, 'reflexion_enabled', _env_bool('AIMO3_REFLEXION_ENABLED', True))
setattr(CFG, 'reflexion_min_votes', _env_int('AIMO3_REFLEXION_MIN_VOTES', 3))
setattr(CFG, 'verifier_enabled', _env_bool('AIMO3_VERIFIER_ENABLED', True))
setattr(CFG, 'vote_entropy_floor', _env_float('AIMO3_VOTE_ENTROPY_FLOOR', 1e-6))

# Fallback tool lite para conter custo em tentativas tardias
setattr(CFG, 'tool_lite_enabled', _env_bool('AIMO3_TOOL_LITE_ENABLED', True))
setattr(CFG, 'tool_lite_after_attempt', _env_int('AIMO3_TOOL_LITE_AFTER_ATTEMPT', 6))
setattr(CFG, 'tool_lite_turns', _env_int('AIMO3_TOOL_LITE_TURNS', 48))

# LoRA/vLLM (serving) — treino permanece opcional
setattr(CFG, 'lora_enabled', _env_bool('AIMO3_LORA_ENABLED', True))
setattr(CFG, 'lora_train_enabled', _env_bool('AIMO3_LORA_TRAIN_ENABLED', False))
setattr(CFG, 'lora_adapter_dir', os.getenv('AIMO3_LORA_ADAPTER_DIR', '/kaggle/working/aimo3_lora_adapters'))
setattr(CFG, 'lora_adapter_name', os.getenv('AIMO3_LORA_ADAPTER_NAME', 'aimo3_lora'))
setattr(CFG, 'lora_max_rank', _env_int('AIMO3_LORA_MAX_RANK', 16))
setattr(CFG, 'lora_rank', _env_int('AIMO3_LORA_RANK', 8))
setattr(CFG, 'lora_alpha', _env_int('AIMO3_LORA_ALPHA', 16))
setattr(CFG, 'lora_dropout', _env_float('AIMO3_LORA_DROPOUT', 0.05))

# ---------------------------------------------------------------------------
# LoRA download público — padrões inteligentes
#
# Quando LoRA está habilitado e nenhum adapter local foi especificado
# explicitamente, habilita o download automático do repositório público
# correspondente ao modelo carregado.  O usuário pode sobrescrever qualquer
# flag individualmente; esta seção apenas define os padrões operacionais para
# que o fluxo funcione sem edição manual no Kaggle.
# ---------------------------------------------------------------------------
_lora_adapter_path_raw = (os.getenv('AIMO3_LORA_ADAPTER_PATH') or '').strip()
_lora_placeholder = _lora_adapter_path_raw in {
    '',
    '/kaggle/input/SEU_DATASET_LORA/aimo3_lora',
    'SEU_DATASET_LORA',
}

if getattr(CFG, 'lora_enabled', False) and _lora_placeholder:
    # Seleciona o repo público correto com base no checkpoint de modelo.
    _model_path_lower = str(getattr(CFG, 'model_path', '') or '').lower()
    if '120b' in _model_path_lower:
        _default_hf_repo = 'sibi-ai/gpt-oss-120b-aimo3-tir-lora'
    elif '20b' in _model_path_lower:
        _default_hf_repo = 'Chenzhiz/aimo3_astral_gpt_oss_lora'
    else:
        _default_hf_repo = 'sibi-ai/gpt-oss-120b-aimo3-tir-lora'

    os.environ.setdefault('AIMO3_LORA_HF_REPO_ID', _default_hf_repo)
    os.environ.setdefault('AIMO3_LORA_AUTO_DOWNLOAD', '1')
    os.environ.setdefault('AIMO3_ALLOW_ONLINE_LORA_DOWNLOAD', '1')
    os.environ.setdefault('AIMO3_LORA_DOWNLOAD_DIR', '/kaggle/working/aimo3_downloaded_lora')
    os.environ.setdefault('AIMO3_LORA_MODULE_NAME', 'aimo3_lora')

    print(f'[STRATEGY] LoRA download público configurado automaticamente:')
    print(f"  repo_id   = {os.environ['AIMO3_LORA_HF_REPO_ID']}")
    print(f"  dest_dir  = {os.environ['AIMO3_LORA_DOWNLOAD_DIR']}")
    print(f"  module    = {os.environ['AIMO3_LORA_MODULE_NAME']}")
    print(f"  download  = AIMO3_LORA_AUTO_DOWNLOAD=1 | AIMO3_ALLOW_ONLINE_LORA_DOWNLOAD=1")
    print(f"  (Defina AIMO3_LORA_ADAPTER_PATH para usar um adapter local e ignorar este fluxo.)")
elif getattr(CFG, 'lora_enabled', False):
    print(f'[STRATEGY] LoRA com adapter explícito: AIMO3_LORA_ADAPTER_PATH={_lora_adapter_path_raw}')
else:
    print('[STRATEGY] LoRA desabilitado (AIMO3_LORA_ENABLED=0).')

print('[STRATEGY] Runtime config aplicado')
print(
    f"[STRATEGY] attempts={getattr(CFG, 'attempts')}, early_stop={getattr(CFG, 'early_stop')}, "
    f"base_temp={getattr(CFG, 'temperature')}, base_min_p={getattr(CFG, 'min_p')}"
)
print(
    f"[STRATEGY] adaptive_sampling={getattr(CFG, 'use_adaptive_sampling')}, "
    f"tool_lite={getattr(CFG, 'tool_lite_enabled')}, verifier={getattr(CFG, 'verifier_enabled')}"
)
print(
    f"[STRATEGY] lora_enabled={getattr(CFG, 'lora_enabled')}, "
    f"lora_train_enabled={getattr(CFG, 'lora_train_enabled')}"
)


[STRATEGY] LoRA download público configurado automaticamente:
  repo_id   = sibi-ai/gpt-oss-120b-aimo3-tir-lora
  dest_dir  = /kaggle/working/aimo3_downloaded_lora
  module    = aimo3_lora
  download  = AIMO3_LORA_AUTO_DOWNLOAD=1 | AIMO3_ALLOW_ONLINE_LORA_DOWNLOAD=1
  (Defina AIMO3_LORA_ADAPTER_PATH para usar um adapter local e ignorar este fluxo.)
[STRATEGY] Runtime config aplicado
[STRATEGY] attempts=10, early_stop=7, base_temp=0.9, base_min_p=0.03
[STRATEGY] adaptive_sampling=True, tool_lite=True, verifier=True
[STRATEGY] lora_enabled=True, lora_train_enabled=False


In [11]:
from __future__ import annotations

from functools import lru_cache
from urllib.parse import quote

import requests

DOMAIN_CONTEXT_KB = {
    'geometry': {
        'keywords': [
            'triangle', 'square', 'rectangle', 'circle', 'angle', 'perimeter', 'area',
            'tangent', 'parallel', 'perpendicular', 'segment', 'polygon', 'diagonal',
            'similar', 'congruent', 'cyclic',
        ],
        'fallback_pages': [
            'Geometry', 'Triangle', 'Circle', 'Perimeter', 'Area',
            'Similar_triangles', 'Power_of_a_point', 'Cyclic_quadrilateral',
        ],
    },
    'number_theory': {
        'keywords': [
            'integer', 'integers', 'divisor', 'divisors', 'divisible', 'gcd', 'lcm',
            'prime', 'composite', 'remainder', 'mod', 'modulo', 'congruent',
            'coprime', 'totient', 'phi', 'factor', 'multiple',
            'modular', 'residue', 'residues', 'chinese remainder', 'crt',
            'multiplicative order', 'order', 'inverse modulo', 'mod inverse',
            'power modulo', 'modular exponentiation',
        ],
        'fallback_pages': [
            'Number_theory', 'Prime_number', 'Divisibility', 'Modular_arithmetic',
            'Greatest_common_divisor', 'Euclidean_algorithm', 'Euler%27s_theorem',
            'Euler%27s_totient_function', 'Modular_exponentiation',
            'Chinese_remainder_theorem', 'Multiplicative_order', 'Carmichael_function',
        ],
    },
    'algebra': {
        'keywords': [
            'equation', 'polynomial', 'function', 'coefficient', 'root', 'quadratic',
            'congruence', 'modulo', 'mod', 'residue class',
        ],
        'fallback_pages': ['Algebra', 'Polynomial', 'Function_(mathematics)', 'Congruence_relation'],
    },
    'combinatorics': {
        'keywords': ['count', 'ways', 'permutation', 'combination', 'choose', 'arrangement'],
        'fallback_pages': ['Combinatorics', 'Permutation', 'Combination'],
    },
}

THEOREM_GUIDANCE_KB = {
    'geometry': [
        {
            'title': 'Angle chasing and triangle structure',
            'match_any': ['triangle', 'angle', 'parallel', 'perpendicular', 'segment', 'diagonal'],
            'guidance': [
                'Write all angle equalities explicitly before computing anything.',
                'Search for similar or congruent triangles as soon as repeated angles or proportional sides appear.',
                'Convert perimeter and length statements into segment equations after the configuration is clear.',
            ],
        },
        {
            'title': 'Circle and cyclic quadrilateral theorems',
            'match_any': ['circle', 'tangent', 'arc', 'cyclic', 'chord'],
            'guidance': [
                'Test whether four points are cyclic and use opposite angles summing to 180 degrees.',
                'Use tangent-chord, equal power, and secant relations only when the configuration supports them.',
                'Translate arc information into angle information before solving for lengths or areas.',
            ],
        },
        {
            'title': 'Area, perimeter, and coordinate fallback',
            'match_any': ['square', 'rectangle', 'perimeter', 'area', 'grid', 'coordinate'],
            'guidance': [
                'If lengths are integral, use divisor structure and parity constraints before brute force.',
                'For rectangles and squares, parameterize side lengths and track which perimeters can repeat or stay distinct.',
                'If synthetic geometry becomes messy, switch to coordinates or algebra while preserving the key invariants.',
            ],
        },
    ],
    'number_theory': [
        {
            'title': 'Euler theorem and totient workflow',
            'match_any': ['mod', 'modulo', 'congruent', 'remainder', 'coprime', 'totient', 'phi'],
            'guidance': [
                'Before using Euler theorem, prove gcd(a, n) = 1 and compute phi(n) from the factorization of n.',
                'Reduce exponents modulo phi(n) only after coprimality is justified.',
                'If gcd(a, n) is not 1, split the modulus or use CRT, valuation arguments, or direct divisibility instead.',
            ],
        },
        {
            'title': 'Divisibility and gcd invariants',
            'match_any': ['divisor', 'divisible', 'gcd', 'lcm', 'factor', 'multiple'],
            'guidance': [
                'Rewrite verbal constraints as exact divisibility statements.',
                'Use gcd/lcm identities and the Euclidean algorithm when the same factors appear repeatedly.',
                'Track parity and residue classes early to eliminate impossible branches.',
            ],
        },
        {
            'title': 'Congruence classes and constructive checking',
            'match_any': ['integer', 'integers', 'prime', 'remainder', 'congruent'],
            'guidance': [
                'Partition the argument by residue classes and preserve the modulus throughout the proof.',
                'Use small examples only to guess a pattern, then prove the pattern rigorously.',
                'When a formula is guessed, test prime powers or small edge cases before trusting it.',
            ],
        },
        {
            'title': 'Modular exponentiation, order, and CRT workflow',
            'match_any': [
                'modulo', 'mod', 'congruent', 'remainder', 'power', 'exponent',
                'chinese remainder', 'crt', 'order', 'inverse modulo',
            ],
            'guidance': [
                'Use repeated squaring (`pow(a, b, n)`) for large modular powers and validate periodicity.',
                'Reduce exponents via phi(n) or multiplicative order only when coprimality is proved.',
                'For composite moduli, split into prime-power congruences, solve each piece, then recombine with CRT.',
                'When gcd(a, n) != 1, avoid blind exponent reduction; use valuation or direct divisibility structure.',
            ],
        },
    ],
}

MODULAR_GUIDANCE_FROM_LECTURES = [
    {
        'title': 'Lecture note pattern: coprimality gate before Euler reduction',
        'source': 'mit6_1200j_s24_lec09.pdf',
        'match_any': ['mod', 'modulo', 'phi', 'totient', 'euler', 'power', 'exponent'],
        'guidance': [
            'Never reduce exponents by phi(n) until gcd(base, n)=1 is justified.',
            'If gcd(base, n) != 1, split the problem by prime factors of n and reason with valuations.',
            'Prefer deriving the period/order explicitly over guessing cycles from few samples.',
        ],
    },
    {
        'title': 'Discrete Math strategy: canonical congruence decomposition',
        'source': 'DiscMathLecture08.pdf',
        'match_any': ['congruent', 'residue', 'mod', 'modulo', 'remainder'],
        'guidance': [
            'Rewrite every statement into explicit congruence equations first.',
            'Normalize to smallest useful modulus and keep transformations reversible.',
            'Track residue classes and parity together to prune impossible branches early.',
        ],
    },
    {
        'title': 'Session method: CRT recombination discipline',
        'source': 'mit6_042js15_session13.pdf',
        'match_any': ['crt', 'chinese remainder', 'modulo', 'simultaneous', 'system'],
        'guidance': [
            'Factor modulus into coprime parts, solve each congruence component, then recombine with CRT.',
            'After recombination, always verify by substitution into original congruences.',
            'Use constructive CRT form to avoid arithmetic slips in final remainder class.',
        ],
    },
    {
        'title': 'Lecture reinforcement: multiplicative order and cycle checks',
        'source': 'DiscMathLecture08 (1).pdf',
        'match_any': ['order', 'period', 'power', 'mod', 'primitive', 'inverse'],
        'guidance': [
            'Compute or bound multiplicative order to control huge exponents.',
            'Use order divisibility facts to reject inconsistent candidate residues.',
            'Cross-check computed order against phi(n) divisibility as a sanity step.',
        ],
    },
]

LOCAL_CONTEXT_LIBRARY = {
    'geometry': [
        {
            'title': 'Geometry fallback knowledge',
            'extract': (
                'Use geometric invariants first: equal angles, similar triangles, cyclic configurations, '
                'and perimeter/area identities. Convert geometry to algebra only after the diagram structure '
                'is understood.'
            ),
            'source': 'local_context',
        }
    ],
    'number_theory': [
        {
            'title': 'Number theory fallback knowledge',
            'extract': (
                'Factor the modulus, isolate gcd constraints, reason in congruence classes, and use Euler theorem '
                'only when coprimality is proved. Prefer exact divisibility arguments over heuristic shortcuts. '
                'For large exponents, use multiplicative order and CRT decomposition when applicable.'
            ),
            'source': 'local_context',
        }
    ],
    'general': [
        {
            'title': 'General fallback knowledge',
            'extract': (
                'Break the problem into invariants, test small cases to detect structure, then prove the pattern '
                'with a valid theorem or exact algebraic argument.'
            ),
            'source': 'local_context',
        }
    ],
}

WIKIPEDIA_HEADERS = {
    'User-Agent': 'AIMO3-Context/1.0 (Kaggle notebook; educational use)',
    'Accept': 'application/json',
}


def _cfg_value(cfg, name: str, default):
    if cfg is None:
        return default
    return getattr(cfg, name, default)


def _truncate_text(text: str, max_chars: int) -> str:
    text = (text or '').strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3].rstrip() + '...'


def _is_modular_domain(domains: list[str]) -> bool:
    return any(domain in {'number_theory', 'algebra'} for domain in (domains or []))


def detect_modular_focus(problem_text: str) -> bool:
    lowered = (problem_text or '').lower()
    modular_keywords = [
        'mod', 'modulo', 'congruent', 'remainder', 'residue',
        'chinese remainder', 'crt', 'totient', 'phi', 'euler',
        'multiplicative order', 'order', 'inverse modulo', 'power modulo',
    ]
    return any(keyword in lowered for keyword in modular_keywords)


def detect_problem_domains(problem_text: str, top_k: int = 2) -> list[str]:
    lowered = (problem_text or '').lower()
    scores = []

    for domain, spec in DOMAIN_CONTEXT_KB.items():
        score = sum(1 for keyword in spec['keywords'] if keyword in lowered)
        if score > 0:
            scores.append((domain, score))

    scores.sort(key=lambda item: item[1], reverse=True)
    return [domain for domain, _ in scores[:top_k]] or ['general']


def _get_wme_session():
    session = globals().get('wme_session')
    if session is not None:
        return session

    access_token = globals().get('access_token')
    if access_token:
        session = requests.Session()
        session.headers.update({
            'Authorization': f'Bearer {access_token}',
            'Accept': 'application/json',
            'User-Agent': 'AIMO3-Context/1.0 (Kaggle notebook; educational use)',
        })
        globals()['wme_session'] = session
        return session

    return None


@lru_cache(maxsize=256)
def wikipedia_summary(page_name: str):
    url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{quote(page_name)}'
    try:
        response = requests.get(url, headers=WIKIPEDIA_HEADERS, timeout=10)
        if response.status_code == 200:
            payload = response.json()
            extract = (payload.get('extract') or '').strip()
            if extract:
                return {
                    'title': payload.get('title', page_name),
                    'extract': extract,
                    'source': 'wikipedia_public',
                }
    except Exception:
        pass
    return None


def wikimedia_enterprise_search(query: str, limit: int = 2) -> list[dict]:
    session = _get_wme_session()
    if session is None:
        return []

    endpoints = [
        'https://on-demand.wikimedia.org/v1/articles/search',
        'https://api.enterprise.wikimedia.com/v2/articles/search',
    ]

    for endpoint in endpoints:
        try:
            response = session.get(endpoint, params={'q': query, 'limit': limit}, timeout=10)
            if response.status_code != 200:
                continue
            if 'application/json' not in response.headers.get('Content-Type', ''):
                continue

            payload = response.json()
            articles = payload.get('articles', []) if isinstance(payload, dict) else []
            items = []
            for article in articles[:limit]:
                title = article.get('title') or article.get('name') or 'Unknown'
                extract = article.get('extract') or article.get('description') or ''
                if extract:
                    items.append({
                        'title': title,
                        'extract': extract,
                        'source': 'wikimedia_enterprise',
                    })
            if items:
                return items
        except Exception:
            continue

    return []


def build_theorem_guidance(problem_text: str, domains: list[str], max_blocks: int = 4) -> list[dict]:
    lowered = (problem_text or '').lower()
    blocks = []
    seen_titles = set()

    for domain in domains:
        for theorem in THEOREM_GUIDANCE_KB.get(domain, []):
            match_any = theorem.get('match_any', [])
            if match_any and not any(keyword in lowered for keyword in match_any):
                continue
            title = theorem['title']
            if title in seen_titles:
                continue
            seen_titles.add(title)
            blocks.append({
                'title': title,
                'guidance': list(theorem['guidance']),
                'domain': domain,
            })

    return blocks[:max_blocks]


def build_modular_guidance(problem_text: str, max_blocks: int = 3) -> list[dict]:
    lowered = (problem_text or '').lower()
    blocks = []
    seen_titles = set()

    for item in MODULAR_GUIDANCE_FROM_LECTURES:
        match_any = item.get('match_any', [])
        if match_any and not any(keyword in lowered for keyword in match_any):
            continue
        title = item['title']
        if title in seen_titles:
            continue
        seen_titles.add(title)
        blocks.append({
            'title': title,
            'guidance': list(item['guidance']),
            'source': item['source'],
        })

    if not blocks:
        fallback = MODULAR_GUIDANCE_FROM_LECTURES[0]
        blocks.append({
            'title': fallback['title'],
            'guidance': list(fallback['guidance']),
            'source': fallback['source'],
        })

    return blocks[:max_blocks]


def build_local_context(domains: list[str], max_items: int = 4) -> list[dict]:
    items = []
    for domain in domains or ['general']:
        items.extend(LOCAL_CONTEXT_LIBRARY.get(domain, []))
    if not items:
        items.extend(LOCAL_CONTEXT_LIBRARY['general'])
    return items[:max_items]


def build_problem_knowledge(problem_text: str, cfg=None) -> dict:
    max_domains = _cfg_value(cfg, 'max_context_domains', 2)
    max_items = _cfg_value(cfg, 'max_context_items', 4)
    max_chars_per_item = _cfg_value(cfg, 'max_context_chars_per_item', 450)
    max_theorem_blocks = _cfg_value(cfg, 'max_theorem_blocks', 4)
    max_modular_blocks = _cfg_value(cfg, 'max_modular_guidance_blocks', 3)
    query_chars = _cfg_value(cfg, 'context_query_chars', 180)

    domains = detect_problem_domains(problem_text, top_k=max_domains)
    query = _truncate_text(problem_text, query_chars)

    compat_mode_47 = _cfg_value(cfg, 'compat_mode_47', False)
    modular_focus_mode = _cfg_value(cfg, 'modular_focus_mode', False)
    modular_focus_detected = detect_modular_focus(problem_text)

    skip_general_context = compat_mode_47 and modular_focus_mode and _is_modular_domain(domains)

    context_items = []
    source = 'no_context'

    if not skip_general_context and _cfg_value(cfg, 'enable_wikimedia_context', True):
        context_items = wikimedia_enterprise_search(query, limit=min(2, max_items))
        if context_items:
            source = 'wikimedia_enterprise'

    if not skip_general_context and not context_items:
        for domain in domains:
            for page_name in DOMAIN_CONTEXT_KB.get(domain, {}).get('fallback_pages', [])[:max_items]:
                summary = wikipedia_summary(page_name)
                if summary and summary.get('extract'):
                    context_items.append(summary)
                if len(context_items) >= max_items:
                    break
            if len(context_items) >= max_items:
                break
        if context_items:
            source = 'wikipedia_public'

    if skip_general_context:
        source = 'compat_plain_prompt'

    if not skip_general_context and not context_items:
        context_items = build_local_context(domains, max_items=max_items)
        source = 'local_context'

    normalized_items = []
    for item in context_items[:max_items]:
        extract = _truncate_text(item.get('extract', ''), max_chars_per_item)
        if extract:
            normalized_items.append({
                'title': item.get('title', 'Untitled'),
                'extract': extract,
                'source': item.get('source', source),
            })

    theorem_guidance = []
    if _cfg_value(cfg, 'enable_theorem_guidance', True):
        theorem_guidance = build_theorem_guidance(problem_text, domains, max_blocks=max_theorem_blocks)

    modular_guidance = []
    if _cfg_value(cfg, 'enable_modular_guidance', True) and _is_modular_domain(domains):
        modular_guidance = build_modular_guidance(problem_text, max_blocks=max_modular_blocks)

    return {
        'domains': domains,
        'source': source,
        'context_items': normalized_items,
        'theorem_guidance': theorem_guidance,
        'modular_guidance': modular_guidance,
        'modular_focus_detected': modular_focus_detected,
        'modular_focus_applied': bool(modular_guidance),
        'context_titles': [item['title'] for item in normalized_items],
        'theorem_titles': [item['title'] for item in theorem_guidance],
        'modular_titles': [item['title'] for item in modular_guidance],
        'modular_sources': sorted({item.get('source', 'unknown') for item in modular_guidance}),
    }


def build_solver_problem_prompt(problem_text: str, knowledge: dict, preference_prompt: str, cfg=None) -> str:
    domains = knowledge.get('domains', [])
    theorem_guidance = knowledge.get('theorem_guidance', [])
    modular_guidance = knowledge.get('modular_guidance', [])

    compat_mode_47 = _cfg_value(cfg, 'compat_mode_47', False)
    modular_focus_mode = _cfg_value(cfg, 'modular_focus_mode', False)

    is_modular_problem = _is_modular_domain(domains) and knowledge.get('modular_focus_detected', False)

    # Modo enxuto para aproximar do notebook 47-50, com foco em aritmética modular.
    if compat_mode_47 and modular_focus_mode and _is_modular_domain(domains):
        sections = [problem_text.strip()]
        sections.append('[Detected mathematical domains]\n' + ', '.join(domains))

        if theorem_guidance:
            theorem_lines = []
            for block in theorem_guidance:
                theorem_lines.append(f'- {block["title"]}:')
                theorem_lines.extend(f'  * {line}' for line in block.get('guidance', []))
            sections.append('[Number theory theorem guide]\n' + '\n'.join(theorem_lines))

        if modular_guidance:
            modular_lines = []
            for block in modular_guidance:
                src = block.get('source', 'unknown')
                modular_lines.append(f'- {block["title"]} (source: {src}):')
                modular_lines.extend(f'  * {line}' for line in block.get('guidance', []))
            sections.append('[Modular arithmetic lecture guidance]\n' + '\n'.join(modular_lines))

        if preference_prompt:
            sections.append('[Available tools and computation guidance]\n' + preference_prompt.strip())

        sections.append(
            '[Important]\n'
            'For modular arithmetic questions, prove coprimality before exponent reduction, '
            'use CRT decomposition explicitly when modulus is composite, and verify the final residue in the original statement.'
        )

        prompt = '\n\n'.join(section for section in sections if section.strip())
        max_total_chars = _cfg_value(cfg, 'max_problem_context_chars', 1800)
        return _truncate_text(prompt, max_total_chars + len(problem_text) + len(preference_prompt) + 768)

    # Modo geral (enriquecido).
    sections = [problem_text.strip()]

    if domains:
        sections.append('[Detected mathematical domains]\n' + ', '.join(domains))

    if theorem_guidance:
        theorem_lines = []
        for block in theorem_guidance:
            theorem_lines.append(f'- {block["title"]}:')
            theorem_lines.extend(f'  * {line}' for line in block.get('guidance', []))
        sections.append('[Theorem adaptation guide]\n' + '\n'.join(theorem_lines))

    if modular_guidance and is_modular_problem:
        modular_lines = []
        for block in modular_guidance:
            src = block.get('source', 'unknown')
            modular_lines.append(f'- {block["title"]} (source: {src}):')
            modular_lines.extend(f'  * {line}' for line in block.get('guidance', []))
        sections.append('[Modular arithmetic lecture guidance]\n' + '\n'.join(modular_lines))

    context_lines = []
    for index, item in enumerate(knowledge.get('context_items', []), start=1):
        context_lines.append(f'{index}. {item["title"]} [{item.get("source", "local_context")}]')
        context_lines.append(f'   {item["extract"]}')
    if context_lines:
        sections.append('[Reference context]\n' + '\n'.join(context_lines))

    if preference_prompt:
        sections.append('[Available tools and computation guidance]\n' + preference_prompt.strip())

    sections.append(
        '[Important]\n'
        'Adapt each theorem to the exact hypotheses of the problem. If a theorem does not apply, '
        'state the missing condition and switch to a valid argument. Prefer structural proofs and '
        'clean reductions over unsupported guesswork.'
    )

    prompt = '\n\n'.join(section for section in sections if section.strip())
    max_total_chars = _cfg_value(cfg, 'max_problem_context_chars', 1800)
    return _truncate_text(prompt, max_total_chars + len(problem_text) + len(preference_prompt) + 768)


print('✓ Wikimedia + theorem context helpers loaded (with modular lecture guidance).')


✓ Wikimedia + theorem context helpers loaded (with modular lecture guidance).


In [12]:
if 'DOMAIN_CONTEXT_KB' in globals():
    normalized_pages = [
        'Number_theory',
        'Prime_number',
        'Divisibility',
        'Modular_arithmetic',
        'Greatest_common_divisor',
        'Euclidean_algorithm',
        "Euler's_theorem",
        "Euler's_totient_function",
        'Modular_exponentiation',
        'Chinese_remainder_theorem',
        'Multiplicative_order',
        'Carmichael_function',
    ]
    DOMAIN_CONTEXT_KB['number_theory']['fallback_pages'] = normalized_pages
    print('✓ Number theory fallback pages normalized for Wikipedia access (modular set).')


✓ Number theory fallback pages normalized for Wikipedia access (modular set).


In [13]:
from __future__ import annotations

try:
    from openai_harmony import (
        Message,
        ReasoningEffort,
        Role,
        SystemContent,
        ToolNamespaceConfig,
    )
except Exception as exc:
    raise RuntimeError(
        '[TEMPLATE] Dependências do Harmony indisponíveis. Execute a célula 5 antes da célula 8.'
    ) from exc

class AIMO3Template:
    """Constrói a conversa inicial no formato esperado pelo modelo GPT-OSS."""

    def __init__(self) -> None:
        """Inicializa o helper de template sem estado adicional."""
        pass

    def get_system_content(
        self,
        system_prompt: str,
        tool_config: ToolNamespaceConfig,
    ) -> SystemContent:
        """Monta o conteúdo de sistema com identidade, esforço e tools.

        Args:
            system_prompt: Prompt-base que define o comportamento do assistente.
            tool_config: Configuração do namespace de tools disponível.

        Returns:
            Objeto `SystemContent` configurado para a conversa.
        """

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self,
        system_prompt: str,
        user_prompt: str,
        tool_config: ToolNamespaceConfig,
    ) -> list[Message]:
        """Cria a lista inicial de mensagens para uma nova tentativa.

        Args:
            system_prompt: Instruções globais do sistema.
            user_prompt: Enunciado do problema a ser resolvido.
            tool_config: Configuração do tool de Python exposto ao modelo.

        Returns:
            Lista com a mensagem de sistema e a mensagem do usuário.
        """

        system_content = self.get_system_content(system_prompt, tool_config)
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [14]:
from __future__ import annotations

import contextlib
import os
import queue
import re
import threading
import time

try:
    from jupyter_client import KernelManager
except Exception:
    from jupyter_client.manager import KernelManager

class AIMO3Sandbox:
    """Gerencia um kernel Jupyter persistente para executar código auxiliar.

    Cada instância mantém um kernel isolado, pronto para cálculos numéricos
    rápidos durante a resolução dos problemas.
    """

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        """Reserva um bloco de portas para um novo kernel Jupyter."""

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):
        """Inicializa o kernel persistente e pré-carrega bibliotecas matemáticas.

        Args:
            timeout: Tempo padrão de espera para execuções no kernel.
        """

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None

        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:
        """Limpa ANSI codes e remove frames irrelevantes do traceback."""

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:
        """Executa código Python no kernel persistente e retorna a saída textual."""

        client = self._client
        effective_timeout = timeout or self._default_timeout

        msg_id = client.execute(
            code,
            store_history=True,
            allow_stdin=False,
            stop_on_error=False,
        )

        stdout_parts = []
        stderr_parts = []

        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])
                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status' and content.get('execution_state') == 'idle':
                break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self) -> None:
        """Encerra canais e libera recursos do kernel associado."""

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self) -> None:
        """Restaura o estado mínimo do kernel entre tentativas de resolução."""

        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):
        """Garante cleanup defensivo caso a instância seja coletada."""

        self.close()

In [15]:
from __future__ import annotations

import threading

try:
    from openai_harmony import Author, Message, Role, TextContent, ToolNamespaceConfig
except Exception as exc:
    raise RuntimeError(
        '[TOOL] Dependências do Harmony indisponíveis. Execute a célula 5 antes da célula 10.'
    ) from exc

class AIMO3Tool:
    """Encapsula o tool de Python exposto ao modelo durante a inferência."""

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):
        """Inicializa o wrapper de execução de código.

        Args:
            local_jupyter_timeout: Timeout padrão para chamadas ao kernel local.
            tool_prompt: Texto descritivo exibido ao modelo sobre o tool.
            sandbox: Sandbox opcional já inicializado para reuso.
        """

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox

        self._owns_session = sandbox is None

        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self) -> None:
        """Cria a sessão Jupyter sob demanda quando necessário."""

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:
        """Adiciona `print(...)` à última linha quando isso ajuda a mostrar saída."""

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:
        """Retorna a descrição textual do tool para o modelo."""

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        """Monta a configuração do namespace `python` consumida pelo modelo."""

        return ToolNamespaceConfig(
            name='python',
            description=self.instruction,
            tools=[],
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:
        """Empacota a saída do tool como mensagem compatível com o protocolo."""

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:
        """Executa o script solicitado pelo modelo e devolve a resposta do tool."""

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [16]:
from __future__ import annotations

import math
import os
import queue
import re
import subprocess
import sys
import threading
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

try:
    from openai import OpenAI
    from openai_harmony import Conversation, HarmonyEncodingName, Role, load_harmony_encoding
except Exception as exc:
    raise RuntimeError(
        '[SOLVER] Dependências de inferência indisponíveis. Execute a célula 5 antes da célula 11.'
    ) from exc


class AIMO3Solver:
    """Orquestra o fluxo completo de resolução de problemas AIMO3.

    A classe prepara o servidor vLLM, inicializa sandboxes persistentes e
    executa múltiplas tentativas paralelas de resolução para selecionar a
    resposta final mais confiável.
    """

    def __init__(self, cfg: CFG, port: int = 8000):
        """Inicializa o solver, o servidor local e os kernels auxiliares.

        Args:
            cfg: Objeto de configuração global do notebook.
            port: Porta usada pelo servidor OpenAI-compatible do vLLM.
        """

        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_model_weights()
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
            timeout=self.cfg.session_timeout,
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        self.problem_context_cache: dict[str, dict] = {}
        self.last_problem_context: dict | None = None

    def _preload_model_weights(self) -> None:
        """Lê os arquivos do checkpoint para aquecer o page cache do sistema."""

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()

        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:
            """Consome o arquivo em blocos grandes para aquecer cache de disco."""

            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:
        """Inicia o servidor vLLM em modo compatível com a API da OpenAI."""

        cmd = [
            sys.executable,
            '-m',
            'vllm.entrypoints.openai.api_server',
            '--seed',
            str(self.cfg.seed),
            '--model',
            self.cfg.model_path,
            '--served-model-name',
            self.cfg.served_model_name,
            '--tensor-parallel-size',
            '1',
            '--max-num-seqs',
            str(self.cfg.batch_size),
            '--gpu-memory-utilization',
            str(self.cfg.gpu_memory_utilization),
            '--host',
            '0.0.0.0',
            '--port',
            str(self.port),
            '--dtype',
            self.cfg.dtype,
            '--kv-cache-dtype',
            self.cfg.kv_cache_dtype,
            '--max-model-len',
            str(self.cfg.context_tokens),
            '--stream-interval',
            str(self.cfg.stream_interval),
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd,
            stdout=self.log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    def _wait_for_server(self) -> None:
        """Espera o servidor vLLM responder antes de iniciar a inferência."""

        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')

                return
            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:
        """Cria o pool de sandboxes persistentes usado pelas tentativas."""

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox() -> AIMO3Sandbox:
            """Factory local usada para paralelizar a criação dos kernels."""

            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:
        """Extrai uma resposta inteira válida a partir do texto do modelo."""

        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass

        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass

        return None

    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
        """Calcula a entropia média dos tokens observados na geração."""

        if not logprobs_buffer:
            return float('inf')

        total_entropy = 0.0
        token_count = 0

        for top_logprobs_dict in logprobs_buffer:
            if not isinstance(top_logprobs_dict, dict):
                continue

            if not top_logprobs_dict:
                continue

            token_entropy = 0.0

            for _, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)

                if prob > 0:
                    token_entropy -= prob * math.log2(prob)

            total_entropy += token_entropy
            token_count += 1

        if token_count == 0:
            return float('inf')

        return total_entropy / token_count

    def _prepare_problem_prompt(self, problem: str) -> tuple[str, dict]:
        """Enriquece o problema com contexto e teoremas uma vez por problema."""

        if problem in self.problem_context_cache:
            payload = dict(self.problem_context_cache[problem])
        else:
            payload = {
                'domains': ['general'],
                'source': 'no_context',
                'context_items': [],
                'theorem_guidance': [],
                'context_titles': [],
                'theorem_titles': [],
            }
            builder = globals().get('build_problem_knowledge')
            if callable(builder):
                try:
                    payload = builder(problem, self.cfg)
                except Exception as exc:
                    payload = {
                        'domains': ['general'],
                        'source': 'context_builder_error',
                        'context_items': [],
                        'theorem_guidance': [],
                        'context_titles': [],
                        'theorem_titles': [],
                        'error': str(exc)[:200],
                    }
            self.problem_context_cache[problem] = dict(payload)

        renderer = globals().get('build_solver_problem_prompt')
        if callable(renderer):
            try:
                user_input = renderer(problem, payload, self.cfg.preference_prompt, self.cfg)
            except Exception as exc:
                payload = dict(payload)
                payload['source'] = 'prompt_render_error'
                payload['error'] = str(exc)[:200]
                user_input = f'{problem} {self.cfg.preference_prompt}'
        else:
            user_input = f'{problem} {self.cfg.preference_prompt}'

        payload = dict(payload)
        payload['prompt_length'] = len(user_input)
        self.last_problem_context = payload
        return user_input, payload

    def _process_attempt(
        self,
        problem: str,
        system_prompt: str,
        attempt_index: int,
        stop_event: threading.Event,
        deadline: float,
    ) -> dict:
        """Executa uma tentativa completa de resolução para um problema."""

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Answer': None,
                'Python Calls': 0,
                'Python Errors': 0,
                'Response Length': 0,
                'Entropy': float('inf'),
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        logprobs_buffer = []

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt,
                sandbox=sandbox,
            )

            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt,
                problem,
                local_tool.tool_config,
            )

            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = encoding.render_conversation_for_completion(
                    conversation,
                    Role.ASSISTANT,
                )
                max_tokens = self.cfg.context_tokens - len(prompt_ids)

                if max_tokens < self.cfg.buffer_tokens:
                    break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=attempt_seed,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.min_p,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True,
                    },
                )

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)

                            chunk_logprobs = chunk.choices[0].logprobs

                            if chunk_logprobs is not None and chunk_logprobs.top_logprobs:
                                logprobs_buffer.extend(chunk_logprobs.top_logprobs)

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)

                            if answer is not None:
                                final_answer = answer
                                break

                finally:
                    stream.close()

                if final_answer is not None:
                    break

                if not token_buffer:
                    break

                new_messages = encoding.parse_messages_from_completion_tokens(
                    token_buffer,
                    Role.ASSISTANT,
                )
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text

                    if (
                        response_text.startswith('[ERROR]')
                        or 'Traceback' in response_text
                        or 'Error:' in response_text
                    ):
                        python_errors += 1

                    conversation.messages.extend(tool_responses)

        except Exception:
            python_errors += 1

        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        mean_entropy = self._compute_mean_entropy(logprobs_buffer)

        return {
            'Attempt': attempt_index + 1,
            'Response Length': total_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Entropy': mean_entropy,
            'Answer': final_answer,
        }

    def _select_answer(self, detailed_results: list) -> int:
        """Escolhe a resposta final usando votos ponderados por entropia."""

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']

            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)

                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer,
                'votes': answer_votes[answer],
                'score': total_weight,
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((item['answer'], item['votes'], item['score']))

        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Score'])

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)

        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer

    def solve_problem(self, problem: str) -> int:
        """Resolve um problema usando múltiplas tentativas paralelas."""

        print(f'\nProblem: {problem}\n')

        user_input, problem_context = self._prepare_problem_prompt(problem)
        print(
            'Context: '
            f"source={problem_context.get('source', 'no_context')} | "
            f"domains={', '.join(problem_context.get('domains', ['general']))} | "
            f"snippets={len(problem_context.get('context_items', []))} | "
            f"theorems={len(problem_context.get('theorem_guidance', []))}"
        )

        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout

        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for system_prompt, attempt_index in tasks:
                future = executor.submit(
                    self._process_attempt,
                    user_input,
                    system_prompt,
                    attempt_index,
                    stop_event,
                    deadline,
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()

                        for pending_future in futures:
                            pending_future.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)

            self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')

            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')

            return 0

        return self._select_answer(detailed_results)

    def __del__(self):
        """Libera processo do vLLM, arquivo de log e sandboxes restantes."""

        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                except Exception:
                    pass


In [17]:
# ===================== OPTIONAL LoRA TRAINING + SOLVER STRATEGY PATCHES =====================
import contextlib
import gc
import glob
from collections import defaultdict

PUBLIC_LORA_REPO_CANDIDATES = {
    '120b': ['sibi-ai/gpt-oss-120b-aimo3-tir-lora'],
    '20b': ['Chenzhiz/aimo3_astral_gpt_oss_lora'],
}


def _local_env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}


def maybe_train_lora_adapter(cfg: CFG) -> str | None:
    """Treina adapter LoRA curto (opcional) e salva para uso no vLLM.

    Ativado somente com AIMO3_LORA_TRAIN_ENABLED=1.
    Treino usa completion-only masking (loss apenas nos tokens da resposta).
    """
    if not getattr(cfg, 'lora_train_enabled', False):
        print('[LoRA] Treino desativado (AIMO3_LORA_TRAIN_ENABLED=0).')
        return None

    model = None
    trainer = None
    adapter_output_dir = os.path.join(
        getattr(cfg, 'lora_adapter_dir', '/kaggle/working/aimo3_lora_adapters'),
        getattr(cfg, 'lora_adapter_name', 'aimo3_lora'),
    )
    os.makedirs(os.path.dirname(adapter_output_dir), exist_ok=True)

    try:
        import torch
        from datasets import Dataset
        from peft import LoraConfig, get_peft_model
        from transformers import (
            AutoModelForCausalLM,
            AutoTokenizer,
            DataCollatorForSeq2Seq,
            Trainer,
            TrainingArguments,
        )

        reference_csv = os.getenv('AIMO3_REFERENCE_CSV') or globals().get('OFFLINE_PATHS', {}).get('reference_csv')
        if not reference_csv or not os.path.exists(reference_csv):
            print('[LoRA][WARN] reference.csv não encontrado; pulando treino do adapter.')
            return None

        max_samples = int(os.getenv('AIMO3_LORA_TRAIN_SAMPLES', '256'))
        max_steps = int(os.getenv('AIMO3_LORA_MAX_STEPS', '50'))
        train_df = pd.read_csv(reference_csv).head(max_samples)

        prompt_col = next((col for col in ['problem', 'question', 'prompt'] if col in train_df.columns), None)
        answer_col = 'answer' if 'answer' in train_df.columns else None

        if prompt_col is None:
            print('[LoRA][WARN] Coluna de enunciado não encontrada no dataset; pulando treino.')
            return None
        if answer_col is None:
            print('[LoRA][WARN] Coluna de resposta não encontrada; completion-only SFT requer alvo. Pulando treino.')
            return None

        records = []
        for _, row in train_df.iterrows():
            question_text = str(row.get(prompt_col, '')).strip()
            answer_text = str(row.get(answer_col, '')).strip()

            if not question_text or not answer_text:
                continue

            prompt_text = f"Problem:\n{question_text}\n\nAnswer:\n"
            target_text = f"\\boxed{{{answer_text}}}"
            records.append({'prompt': prompt_text, 'target': target_text})

        if len(records) < 8:
            print(f"[LoRA][WARN] Dataset muito pequeno ({len(records)} exemplos); pulando treino.")
            return None

        dataset = Dataset.from_list(records)

        print('[LoRA] Carregando modelo base para fine-tuning LoRA...')
        model = AutoModelForCausalLM.from_pretrained(
            cfg.model_path,
            torch_dtype='auto',
            device_map='auto',
            trust_remote_code=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(cfg.model_path, trust_remote_code=True)

        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token

        candidate_target_modules = [
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'gate_proj', 'up_proj', 'down_proj',
        ]
        available_leaf_modules = {name.split('.')[-1] for name, _ in model.named_modules()}
        target_modules = [name for name in candidate_target_modules if name in available_leaf_modules]
        if not target_modules:
            print('[LoRA][WARN] Nenhum target_module padrão encontrado no modelo; pulando treino.')
            return None

        # GELU activation modules (gate_proj, up_proj) belong to the FFN SwiGLU/GELU block.
        # Confirming their presence ensures the LoRA adapter covers these gating projections.
        for _gelu_m in ['gate_proj', 'up_proj']:
            _gelu_status = 'presente' if _gelu_m in target_modules else 'ausente (nao encontrado nas camadas folha)'
            print(f'[LoRA][GELU] {_gelu_m}: {_gelu_status}')
        print(f'[LoRA] target_modules finais: {target_modules}')

        lora_cfg = LoraConfig(
            r=int(getattr(cfg, 'lora_rank', 8)),
            lora_alpha=int(getattr(cfg, 'lora_alpha', 16)),
            lora_dropout=float(getattr(cfg, 'lora_dropout', 0.05)),
            bias='none',
            task_type='CAUSAL_LM',
            target_modules=target_modules,
        )

        model = get_peft_model(model, lora_cfg)
        model.print_trainable_parameters()

        max_length = int(os.getenv('AIMO3_LORA_TRAIN_MAX_LENGTH', '2048'))

        def _tokenize(batch):
            all_input_ids = []
            all_attention_mask = []
            all_labels = []

            prompts = batch.get('prompt', [])
            targets = batch.get('target', [])

            for prompt_text, target_text in zip(prompts, targets):
                full_text = f"{prompt_text}{target_text}"
                prompt_encoded = tokenizer(
                    prompt_text,
                    truncation=True,
                    max_length=max_length,
                    padding=False,
                    add_special_tokens=False,
                )
                full_encoded = tokenizer(
                    full_text,
                    truncation=True,
                    max_length=max_length,
                    padding=False,
                    add_special_tokens=False,
                )

                input_ids = list(full_encoded['input_ids'])
                attention_mask = list(full_encoded['attention_mask'])
                prompt_len = min(len(prompt_encoded['input_ids']), len(input_ids))

                if tokenizer.eos_token_id is not None and (not input_ids or input_ids[-1] != tokenizer.eos_token_id):
                    input_ids.append(tokenizer.eos_token_id)
                    attention_mask.append(1)

                labels = [-100] * prompt_len + input_ids[prompt_len:]
                if not any(label != -100 for label in labels):
                    continue

                all_input_ids.append(input_ids)
                all_attention_mask.append(attention_mask)
                all_labels.append(labels)

            return {
                'input_ids': all_input_ids,
                'attention_mask': all_attention_mask,
                'labels': all_labels,
            }

        tokenized_dataset = dataset.map(_tokenize, batched=True, remove_columns=['prompt', 'target'])
        if len(tokenized_dataset) < 8:
            print(f"[LoRA][WARN] Dataset tokenizado insuficiente ({len(tokenized_dataset)} exemplos).")
            return None

        training_args = TrainingArguments(
            output_dir=adapter_output_dir,
            overwrite_output_dir=True,
            max_steps=max_steps,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            learning_rate=float(os.getenv('AIMO3_LORA_LR', '2e-4')),
            logging_steps=5,
            save_steps=max(10, max_steps // 2),
            bf16=True,
            fp16=False,
            report_to=[],
            remove_unused_columns=False,
        )

        data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model,
            padding=True,
            label_pad_token_id=-100,
            return_tensors='pt',
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset,
            data_collator=data_collator,
        )

        print(f"[LoRA] Iniciando treino rápido ({max_steps} steps, completion-only masking)...")
        trainer.train()

        model.save_pretrained(adapter_output_dir)
        tokenizer.save_pretrained(adapter_output_dir)
        print(f"[LoRA] Adapter salvo em: {adapter_output_dir}")
        return adapter_output_dir

    except Exception as exc:
        print(f"[LoRA][WARN] Falha no treino LoRA: {exc}")
        return None
    finally:
        with contextlib.suppress(Exception):
            del trainer
        with contextlib.suppress(Exception):
            del model
        gc.collect()
        if 'torch' in globals():
            with contextlib.suppress(Exception):
                torch.cuda.empty_cache()


_trained_lora_dir = maybe_train_lora_adapter(CFG)
if _trained_lora_dir:
    print(f"[LoRA] Adapter treinado disponível em: {_trained_lora_dir}")


def _infer_public_lora_repo_candidates(cfg: CFG) -> list[str]:
    repo_candidates = []
    explicit_repo = (os.getenv('AIMO3_LORA_HF_REPO_ID') or '').strip()
    if explicit_repo:
        repo_candidates.append(explicit_repo)

    model_path = str(getattr(cfg, 'model_path', '') or '').lower()
    if '120b' in model_path:
        repo_candidates.extend(PUBLIC_LORA_REPO_CANDIDATES['120b'])
    if '20b' in model_path:
        repo_candidates.extend(PUBLIC_LORA_REPO_CANDIDATES['20b'])

    deduped = []
    seen = set()
    for repo_id in repo_candidates:
        if repo_id and repo_id not in seen:
            deduped.append(repo_id)
            seen.add(repo_id)
    return deduped


def _inspect_adapter_dir(candidate: str | None) -> dict:
    report = {
        'path': candidate,
        'exists': False,
        'is_dir': False,
        'has_adapter_config': False,
        'adapter_weights': [],
        'valid': False,
        'reason': 'empty_candidate',
    }

    if not candidate:
        return report

    report['exists'] = os.path.exists(candidate)
    if not report['exists']:
        report['reason'] = 'missing_path'
        return report

    report['is_dir'] = os.path.isdir(candidate)
    if not report['is_dir']:
        report['reason'] = 'not_directory'
        return report

    report['has_adapter_config'] = os.path.exists(os.path.join(candidate, 'adapter_config.json'))
    report['adapter_weights'] = sorted(glob.glob(os.path.join(candidate, 'adapter_model*')))
    report['valid'] = report['has_adapter_config'] or bool(report['adapter_weights'])
    report['reason'] = 'ok' if report['valid'] else 'missing_adapter_files'
    return report


def _materialize_public_lora_adapter(cfg: CFG) -> str | None:
    cached_dir = globals().get('_downloaded_lora_dir')
    cached_report = _inspect_adapter_dir(cached_dir)
    if cached_report['valid']:
        return cached_dir

    repo_candidates = _infer_public_lora_repo_candidates(cfg)
    if not repo_candidates:
        return None

    auto_download = _local_env_bool('AIMO3_LORA_AUTO_DOWNLOAD', False)
    if not auto_download:
        return None

    allow_online_download = _local_env_bool('AIMO3_ALLOW_ONLINE_LORA_DOWNLOAD', False)
    if allow_online_download:
        os.environ['HF_HUB_OFFLINE'] = '0'
        os.environ['TRANSFORMERS_OFFLINE'] = '0'

    try:
        from huggingface_hub import snapshot_download
    except Exception as exc:
        print(f'[LoRA][WARN] huggingface_hub indisponível para baixar adapter público: {exc}')
        return None

    download_root = (os.getenv('AIMO3_LORA_DOWNLOAD_DIR') or '/kaggle/working/aimo3_downloaded_lora').strip()
    os.makedirs(download_root, exist_ok=True)

    allow_patterns = [
        'adapter_config.json',
        'adapter_model*',
        'chat_template.jinja',
        'special_tokens_map.json',
        'tokenizer.json',
        'tokenizer_config.json',
        'added_tokens.json',
        'README.md',
    ]

    errors = []
    for repo_id in repo_candidates:
        local_dir = os.path.join(download_root, repo_id.split('/')[-1])
        local_report = _inspect_adapter_dir(local_dir)
        if local_report['valid']:
            os.environ['AIMO3_LORA_ADAPTER_PATH'] = local_dir
            globals()['_downloaded_lora_dir'] = local_dir
            print(f'[LoRA] Adapter público já materializado em cache: {local_dir}')
            return local_dir

        try:
            print(f'[LoRA] Materializando adapter público: {repo_id}')
            snapshot_path = snapshot_download(
                repo_id=repo_id,
                local_dir=local_dir,
                allow_patterns=allow_patterns,
                local_files_only=not allow_online_download,
            )
            snapshot_report = _inspect_adapter_dir(snapshot_path)
            if snapshot_report['valid']:
                os.environ['AIMO3_LORA_ADAPTER_PATH'] = snapshot_path
                globals()['_downloaded_lora_dir'] = snapshot_path
                print(f'[LoRA] Adapter público pronto em: {snapshot_path}')
                return snapshot_path
            errors.append(f'{repo_id}: download concluído, mas sem artefatos válidos de adapter')
        except Exception as exc:
            errors.append(f'{repo_id}: {exc}')

    if errors:
        print('[LoRA][WARN] Falha ao materializar adapter público:')
        for item in errors:
            print(f'  - {item}')

    return None


def _build_lora_resolution_candidates(cfg: CFG) -> list[tuple[str, str | None]]:
    adapter_name = str(getattr(cfg, 'lora_adapter_name', 'aimo3_lora'))
    configured_root = str(getattr(cfg, 'lora_adapter_dir', '/kaggle/working/aimo3_lora_adapters'))

    candidates = [
        ('AIMO3_LORA_ADAPTER_PATH', (os.getenv('AIMO3_LORA_ADAPTER_PATH') or '').strip()),
        ('_trained_lora_dir', globals().get('_trained_lora_dir')),
        ('_downloaded_lora_dir', globals().get('_downloaded_lora_dir')),
        ('OFFLINE_PATHS.lora_adapter_path', globals().get('OFFLINE_PATHS', {}).get('lora_adapter_path')),
        ('AIMO3_LORA_ADAPTER_DIR + AIMO3_LORA_ADAPTER_NAME', os.path.join(configured_root, adapter_name)),
    ]

    for match in sorted(glob.glob(f'/kaggle/input/*/{adapter_name}')):
        candidates.append(('/kaggle/input/*/{adapter_name}', match))
    for match in sorted(glob.glob(f'/kaggle/input/*/*/{adapter_name}')):
        candidates.append(('/kaggle/input/*/*/{adapter_name}', match))
    for match in sorted(glob.glob('/kaggle/input/*/*lora*')):
        candidates.append(('/kaggle/input/*/*lora*', match))
    for match in sorted(glob.glob('/kaggle/input/*/*/*lora*')):
        candidates.append(('/kaggle/input/*/*/*lora*', match))
    for match in sorted(glob.glob('/kaggle/working/*lora*')):
        candidates.append(('/kaggle/working/*lora*', match))

    deduped = []
    seen = set()
    for label, path in candidates:
        normalized = (path or '').strip()
        key = normalized.lower()
        if key in seen:
            continue
        seen.add(key)
        deduped.append((label, normalized or None))
    return deduped


def _print_lora_resolution_diagnostics(cfg: CFG, candidates: list[tuple[str, str | None]]) -> None:
    print('[LoRA][DIAG] Nenhum adapter válido foi encontrado. Varredura de candidatos:')
    for label, path in candidates:
        report = _inspect_adapter_dir(path)
        adapter_weights_count = len(report['adapter_weights'])
        display_path = path or '<vazio>'
        print(
            f"  - source={label} | path={display_path} | exists={report['exists']} | "
            f"is_dir={report['is_dir']} | adapter_config={report['has_adapter_config']} | "
            f"adapter_model_files={adapter_weights_count} | reason={report['reason']}"
        )

    public_candidates = _infer_public_lora_repo_candidates(cfg)
    if public_candidates:
        print('[LoRA][DIAG] Repositórios públicos candidatos para materialização:')
        for repo_id in public_candidates:
            print(f'  - {repo_id}')
        print('[LoRA][DIAG] Para baixar automaticamente no ambiente com internet habilitada:')
        print('  AIMO3_LORA_HF_REPO_ID=<repo_id>')
        print('  AIMO3_LORA_AUTO_DOWNLOAD=1')
        print('  AIMO3_ALLOW_ONLINE_LORA_DOWNLOAD=1')


def _resolve_lora_adapter_dir(cfg: CFG) -> str | None:
    """Resolve caminho do adapter com fallback para env, treino local, Kaggle input e HF público."""
    public_adapter_dir = _materialize_public_lora_adapter(cfg)
    if public_adapter_dir:
        public_report = _inspect_adapter_dir(public_adapter_dir)
        if public_report['valid']:
            return public_adapter_dir

    candidates = _build_lora_resolution_candidates(cfg)
    for _, candidate in candidates:
        report = _inspect_adapter_dir(candidate)
        if report['valid']:
            return candidate

    _print_lora_resolution_diagnostics(cfg, candidates)
    return None


# --------------------- PATCHES DE ESTRATÉGIA NO SOLVER ---------------------
_orig_start_server = AIMO3Solver._start_server
_orig_process_attempt = AIMO3Solver._process_attempt
_orig_select_answer = AIMO3Solver._select_answer
_orig_prepare_problem_prompt = AIMO3Solver._prepare_problem_prompt
_orig_scan_for_answer = AIMO3Solver._scan_for_answer


def _patched_start_server(self) -> subprocess.Popen:
    """Inicia vLLM com suporte opcional a LoRA quando adapter existir.

    Importante: em API OpenAI do vLLM, o `model` da request deve ser o nome do módulo LoRA
    quando o adapter está ativo.
    """
    base_model_name = str(getattr(self.cfg, 'served_model_name', 'gpt-oss'))
    cmd = [
        sys.executable,
        '-m',
        'vllm.entrypoints.openai.api_server',
        '--seed', str(self.cfg.seed),
        '--model', self.cfg.model_path,
        '--served-model-name', base_model_name,
        '--tensor-parallel-size', '1',
        '--max-num-seqs', str(self.cfg.batch_size),
        '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
        '--host', '0.0.0.0',
        '--port', str(self.port),
        '--dtype', self.cfg.dtype,
        '--kv-cache-dtype', self.cfg.kv_cache_dtype,
        '--max-model-len', str(self.cfg.context_tokens),
        '--stream-interval', str(self.cfg.stream_interval),
        '--async-scheduling',
        '--disable-log-stats',
        '--enable-prefix-caching',
    ]

    lora_module_name = os.getenv(
        'AIMO3_LORA_MODULE_NAME',
        str(getattr(self.cfg, 'lora_adapter_name', 'aimo3_lora'))
    )
    adapter_dir = _resolve_lora_adapter_dir(self.cfg)
    use_lora = bool(getattr(self.cfg, 'lora_enabled', False) and adapter_dir)

    if use_lora:
        cmd.extend([
            '--enable-lora',
            '--max-lora-rank', str(getattr(self.cfg, 'lora_max_rank', 16)),
            '--lora-modules', f"{lora_module_name}={adapter_dir}",
        ])
        print(f"[SOLVER] LoRA ativo no vLLM: module={lora_module_name} | dir={adapter_dir}")
    else:
        explicit_path = (os.getenv('AIMO3_LORA_ADAPTER_PATH') or '').strip() or '<não definido>'
        public_repo = (os.getenv('AIMO3_LORA_HF_REPO_ID') or '').strip() or '<auto-sugestão>'
        print(
            '[SOLVER] Rodando sem LoRA (adapter não encontrado ou desativado). '
            f'explicit_path={explicit_path} | hf_repo={public_repo} | '
            f'auto_download={_local_env_bool("AIMO3_LORA_AUTO_DOWNLOAD", False)}'
        )

    active_generation_model = lora_module_name if use_lora else base_model_name
    setattr(self, 'lora_active', use_lora)
    setattr(self, 'lora_module_name', lora_module_name)
    setattr(self, 'lora_adapter_dir', adapter_dir)
    setattr(self, 'active_generation_model', active_generation_model)

    # `_process_attempt` original usa `self.cfg.served_model_name` em completions.create.
    # Aqui apontamos para o alias LoRA quando ativo.
    self.cfg.served_model_name = active_generation_model
    print(f"[SOLVER] Modelo efetivo para geração: {self.cfg.served_model_name}")

    self.log_file = open('vllm_server.log', 'w')
    return subprocess.Popen(
        cmd,
        stdout=self.log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )


def _patched_prepare_problem_prompt(self, problem: str) -> tuple[str, dict]:
    user_input, payload = _orig_prepare_problem_prompt(self, problem)

    lowered = (problem or '').lower()
    strategy_hint = None
    if any(term in lowered for term in ['mod', 'modulo', 'congruent', 'remainder', 'crt']):
        strategy_hint = (
            'Use modular arithmetic workflow: prove coprimality before exponent reduction, '
            'apply CRT on composite modulus when needed, and validate final residue.'
        )
    elif any(term in lowered for term in ['triangle', 'circle', 'angle', 'area', 'perimeter']):
        strategy_hint = (
            'Use geometric invariants first: angle-chasing, similarity/congruence, then algebraic reduction.'
        )
    elif any(term in lowered for term in ['count', 'ways', 'permutation', 'combination']):
        strategy_hint = (
            'Use combinatorial case split + counting invariants; verify against small edge cases.'
        )

    if strategy_hint:
        user_input = f"{user_input}\n\n[Strategy hint]\n{strategy_hint}"
        payload = dict(payload)
        payload['strategy_hint'] = strategy_hint
        payload['prompt_length'] = len(user_input)
        self.last_problem_context = payload

    return user_input, payload


def _patched_scan_for_answer(self, text: str) -> int | None:
    value = _orig_scan_for_answer(self, text)
    if value is not None:
        return value

    numeric_candidates = re.findall(r'(?<!\d)(\d{1,5})(?!\d)', text or '')
    for candidate in reversed(numeric_candidates):
        parsed = int(candidate)
        if 0 <= parsed <= 99999:
            return parsed
    return None


def _patched_process_attempt(
    self,
    problem: str,
    system_prompt: str,
    attempt_index: int,
    stop_event: threading.Event,
    deadline: float,
) -> dict:
    """Aplica SC adaptativo por tentativa (temperatura/min_p/turns)."""
    old_temperature = self.cfg.temperature
    old_min_p = self.cfg.min_p
    old_turns = self.cfg.turns

    try:
        if bool(getattr(self.cfg, 'use_adaptive_sampling', False)):
            adaptive_temps = list(getattr(self.cfg, 'adaptive_temperatures', []))
            adaptive_min_ps = list(getattr(self.cfg, 'adaptive_min_p', []))
            adaptive_turns = list(getattr(self.cfg, 'adaptive_turns', []))
            idx = attempt_index % max(1, len(adaptive_temps) if adaptive_temps else 1)

            if adaptive_temps:
                self.cfg.temperature = adaptive_temps[idx]

            if adaptive_min_ps:
                self.cfg.min_p = adaptive_min_ps[idx % len(adaptive_min_ps)]

            if adaptive_turns:
                self.cfg.turns = adaptive_turns[idx % len(adaptive_turns)]

        if bool(getattr(self.cfg, 'tool_lite_enabled', False)):
            threshold = int(getattr(self.cfg, 'tool_lite_after_attempt', 6))
            if (attempt_index + 1) >= threshold:
                _problem_domains = (getattr(self, 'last_problem_context', None) or {}).get('domains', []) or []
                _complex_domains = {'number_theory', 'algebra'}
                if not _complex_domains.intersection(_problem_domains):
                    self.cfg.turns = min(self.cfg.turns, int(getattr(self.cfg, 'tool_lite_turns', 48)))

        # Garantia defensiva: se o servidor reportar LoRA ativo, usa o alias LoRA nas requests.
        active_generation_model = getattr(self, 'active_generation_model', self.cfg.served_model_name)
        self.cfg.served_model_name = active_generation_model

        return _orig_process_attempt(
            self,
            problem,
            system_prompt,
            attempt_index,
            stop_event,
            deadline,
        )
    finally:
        self.cfg.temperature = old_temperature
        self.cfg.min_p = old_min_p
        self.cfg.turns = old_turns


def _candidate_quality_score(self, result: dict) -> float:
    answer = result.get('Answer')
    if answer is None:
        return float('-inf')

    entropy = float(result.get('Entropy', float('inf')))
    entropy_floor = max(float(getattr(self.cfg, 'vote_entropy_floor', 1e-6)), 1e-12)
    entropy_term = 1.0 / max(entropy, entropy_floor) if math.isfinite(entropy) else 0.0

    python_calls = int(result.get('Python Calls', 0) or 0)
    python_errors = int(result.get('Python Errors', 0) or 0)
    response_length = int(result.get('Response Length', 0) or 0)

    tool_bonus = 0.10 * min(python_calls, 3) - 0.20 * python_errors
    length_bonus = 0.08 if response_length >= 96 else 0.0

    verifier_bonus = 0.0
    if bool(getattr(self.cfg, 'verifier_enabled', False)):
        context_payload = getattr(self, 'last_problem_context', None) or {}
        domains = context_payload.get('domains', []) or []
        with contextlib.suppress(Exception):
            answer_int = int(answer)
            if 0 <= answer_int <= 99999:
                if 'number_theory' in domains:
                    # Modular results rarely exceed 999; penalize suspiciously large values.
                    verifier_bonus += 0.20 if answer_int <= 999 else 0.08
                elif 'geometry' in domains:
                    verifier_bonus += 0.15 if answer_int <= 9999 else 0.06
                elif 'combinatorics' in domains:
                    verifier_bonus += 0.15
                else:
                    verifier_bonus += 0.08
            if 'number_theory' in domains:
                verifier_bonus += 0.03  # extra domain boost regardless of range

    return entropy_term + tool_bonus + length_bonus + verifier_bonus


def _patched_select_answer(self, detailed_results: list) -> int:
    """Voto majoritário + rerank por qualidade/verificação com reflexion barato."""
    answer_scores = defaultdict(float)
    answer_votes = defaultdict(int)

    for result in detailed_results:
        answer = result.get('Answer')
        if answer is None:
            continue

        quality = _candidate_quality_score(self, result)
        answer_scores[answer] += quality
        answer_votes[answer] += 1

    scored_answers = [
        {
            'answer': answer,
            'votes': answer_votes[answer],
            'score': score,
        }
        for answer, score in answer_scores.items()
    ]
    scored_answers.sort(key=lambda item: item['score'], reverse=True)

    vote_dataframe = pd.DataFrame(
        [
            (item['answer'], item['votes'], item['score'])
            for item in scored_answers
        ],
        columns=['Answer', 'Votes', 'Score'],
    )
    if not vote_dataframe.empty:
        vote_dataframe = vote_dataframe.round({'Score': 3})
    display(vote_dataframe)

    if not scored_answers:
        print('\nFinal Answer: 0\n')
        return 0

    final_answer = scored_answers[0]['answer']

    if bool(getattr(self.cfg, 'reflexion_enabled', False)) and len(scored_answers) > 1:
        top_votes = scored_answers[0]['votes']
        min_votes = int(getattr(self.cfg, 'reflexion_min_votes', 3))
        if top_votes < min_votes:
            score_gap = scored_answers[0]['score'] - scored_answers[1]['score']
            if score_gap < 0.08 and scored_answers[1]['votes'] >= scored_answers[0]['votes']:
                print('[RERANK] Reflexion low-cost: second candidate selected by vote tie-break.')
                final_answer = scored_answers[1]['answer']

    print(f'\nFinal Answer: {final_answer}\n')
    return final_answer


AIMO3Solver._start_server = _patched_start_server
AIMO3Solver._prepare_problem_prompt = _patched_prepare_problem_prompt
AIMO3Solver._scan_for_answer = _patched_scan_for_answer
AIMO3Solver._process_attempt = _patched_process_attempt
AIMO3Solver._select_answer = _patched_select_answer

print('[PATCH] AIMO3Solver atualizado com LoRA real (server+model alias), download público opcional, SC adaptativo, rerank/verifier e fallback tool-lite.')

[LoRA] Treino desativado (AIMO3_LORA_TRAIN_ENABLED=0).
[PATCH] AIMO3Solver atualizado com LoRA real (server+model alias), download público opcional, SC adaptativo, rerank/verifier e fallback tool-lite.


In [18]:
# ===================== LoRA COMPAT PATCH (LOCAL-FIRST, PRABH-STYLE) =====================
import os
import glob
import contextlib

if globals().get('_aimo3_lora_local_first_patch_applied', False):
    print('[LoRA][COMPAT] Patch local-first já aplicado; pulando.')
else:
    def _compat_is_valid_adapter_dir(path: str | None) -> bool:
        if not path:
            return False

        inspector = globals().get('_inspect_adapter_dir')
        if callable(inspector):
            with contextlib.suppress(Exception):
                return bool(inspector(path).get('valid', False))

        if not os.path.isdir(path):
            return False

        has_cfg = os.path.exists(os.path.join(path, 'adapter_config.json'))
        has_weights = bool(glob.glob(os.path.join(path, 'adapter_model*')))
        return bool(has_cfg or has_weights)


    def _compat_expand_candidates(path: str) -> list[str]:
        candidates = [path]
        if os.path.isdir(path):
            candidates.extend(sorted(glob.glob(os.path.join(path, '*'))))
            candidates.extend(sorted(glob.glob(os.path.join(path, '*', '*'))))
        return candidates


    def _compat_find_local_adapter() -> str | None:
        explicit_path = (os.getenv('AIMO3_LORA_ADAPTER_PATH') or '').strip()

        base_candidates = []
        if explicit_path:
            base_candidates.append(explicit_path)

        # Caminhos típicos do fluxo de dataset LoRA em Kaggle.
        base_candidates.extend([
            '/kaggle/input/aimo3-lora-adapters',
            '/kaggle/input/aimo3-math-adapter',
            '/kaggle/working/aimo3_lora_adapters',
        ])

        # Varredura ampla incluindo raiz e subpastas (1-3 níveis).
        for pattern in [
            '/kaggle/input/*lora*',
            '/kaggle/input/*adapter*',
            '/kaggle/input/*/*lora*',
            '/kaggle/input/*/*adapter*',
            '/kaggle/input/*/*/*lora*',
            '/kaggle/input/*/*/*adapter*',
            '/kaggle/working/*lora*',
            '/kaggle/working/*adapter*',
            '/kaggle/working/*/*lora*',
            '/kaggle/working/*/*adapter*',
        ]:
            base_candidates.extend(sorted(glob.glob(pattern)))

        seen = set()
        for candidate in base_candidates:
            for expanded in _compat_expand_candidates(candidate):
                normalized = (expanded or '').strip()
                key = normalized.lower()
                if not key or key in seen:
                    continue
                seen.add(key)
                if _compat_is_valid_adapter_dir(normalized):
                    return normalized

        return None


    _explicit_raw = (os.getenv('AIMO3_LORA_ADAPTER_PATH') or '').strip()
    _placeholder_path = _explicit_raw in {
        '',
        '/kaggle/input/SEU_DATASET_LORA/aimo3_lora',
        'SEU_DATASET_LORA',
    }

    _local_adapter = _compat_find_local_adapter()

    if getattr(CFG, 'lora_enabled', False) and _local_adapter:
        if _placeholder_path or not _explicit_raw:
            os.environ['AIMO3_LORA_ADAPTER_PATH'] = _local_adapter

        # Se já existe adapter local válido, evita download online por padrão.
        os.environ.setdefault('AIMO3_LORA_AUTO_DOWNLOAD', '0')
        os.environ.setdefault('AIMO3_ALLOW_ONLINE_LORA_DOWNLOAD', '0')

        print('[LoRA][COMPAT] Adapter local válido detectado e priorizado:')
        print(f"  AIMO3_LORA_ADAPTER_PATH={os.environ.get('AIMO3_LORA_ADAPTER_PATH')}")
        print('  AIMO3_LORA_AUTO_DOWNLOAD=0 | AIMO3_ALLOW_ONLINE_LORA_DOWNLOAD=0 (defaults)')
    else:
        print('[LoRA][COMPAT] Nenhum adapter local válido encontrado; mantendo fallback público.')


    if '_build_lora_resolution_candidates' in globals() and '_orig_build_lora_resolution_candidates_compat' not in globals():
        _orig_build_lora_resolution_candidates_compat = _build_lora_resolution_candidates

        def _build_lora_resolution_candidates(cfg: CFG) -> list[tuple[str, str | None]]:
            candidates = _orig_build_lora_resolution_candidates_compat(cfg)

            extras = []
            for pattern in [
                '/kaggle/input/*lora*',
                '/kaggle/input/*adapter*',
                '/kaggle/input/*/*lora*',
                '/kaggle/input/*/*adapter*',
                '/kaggle/input/*/*/*lora*',
                '/kaggle/input/*/*/*adapter*',
                '/kaggle/working/*lora*',
                '/kaggle/working/*adapter*',
                '/kaggle/working/*/*lora*',
                '/kaggle/working/*/*adapter*',
            ]:
                for match in sorted(glob.glob(pattern)):
                    extras.append((pattern, match))

            seen = set((path or '').strip().lower() for _, path in candidates)
            for label, path in extras:
                key = (path or '').strip().lower()
                if key and key not in seen:
                    candidates.append((label, path))
                    seen.add(key)

            return candidates

        globals()['_build_lora_resolution_candidates'] = _build_lora_resolution_candidates
        print('[LoRA][COMPAT] Builder de candidatos ampliado para layouts de dataset local.')


    # ----- TIES MERGE FUNCTION -----
    def _apply_ties_merge(
        dir_a: str,
        dir_b: str,
        output_dir: str | None = None,
        density: float = 0.20,
    ) -> str | None:
        """TIES merging (Trim, Elect-Sign, Disjoint-Merge) de dois adapters LoRA.
        Etapas: 1) Trim: zera abs abaixo do percentil (1-density)*100.
                2) Elect Sign: sinal eleito pela soma dos tensores trimados.
                3) Disjoint Merge: media valores que concordam com o sinal eleito.
        density=0.20 => mantém top-20% dos parâmetros por magnitude."""
        import gc as _gc
        _output_dir = output_dir or '/kaggle/working/aimo3_lora_merged'
        _insp = globals().get('_inspect_adapter_dir')
        if callable(_insp) and _insp(_output_dir).get('valid', False):
            print(f'[TIES] Adapter fundido em cache: {_output_dir}')
            return _output_dir
        try:
            import pathlib as _pathlib
            import shutil as _shutil
            import torch as _torch
            try:
                from safetensors.torch import load_file as _load_sf, save_file as _save_sf
                _use_sf = True
            except ImportError:
                _use_sf = False

            def _load_adapter_tensors(d: str) -> dict:
                p = _pathlib.Path(d)
                if _use_sf:
                    sf_files = sorted(p.glob('adapter_model*.safetensors'))
                    if sf_files:
                        t = {}
                        for f in sf_files:
                            t.update(_load_sf(str(f)))
                        return t
                bin_files = sorted(p.glob('adapter_model*.bin'))
                if not bin_files:
                    raise FileNotFoundError(f'Sem weights de adapter em {d}')
                t = {}
                for f in bin_files:
                    t.update(_torch.load(str(f), map_location='cpu'))
                return t

            def _trim_tensor(x, den):
                """Etapa 1 TIES: zera valores abaixo do limiar de magnitude."""
                if den >= 1.0:
                    return x.clone()
                flat = x.abs().flatten()
                k = max(1, int(flat.numel() * den))
                thr = flat.topk(k, largest=True, sorted=False).values.min()
                return x * (x.abs() >= thr)

            print(f'[TIES] Carregando adapter A: {dir_a}')
            ta_map = _load_adapter_tensors(dir_a)
            print(f'[TIES] Carregando adapter B: {dir_b}')
            tb_map = _load_adapter_tensors(dir_b)
            print(f'[TIES] Iniciando merge ({len(ta_map)} / {len(tb_map)} tensores, density={density})')
            merged = {}
            for key in sorted(set(ta_map) | set(tb_map)):
                ta = ta_map.get(key)
                tb = tb_map.get(key)
                if ta is None:
                    merged[key] = tb.clone(); continue
                if tb is None:
                    merged[key] = ta.clone(); continue
                if ta.shape != tb.shape:
                    merged[key] = ta.clone(); continue  # Incompatible rank: keep A.
                taf = _trim_tensor(ta.float(), density)  # Etapa 1
                tbf = _trim_tensor(tb.float(), density)
                esign = _torch.sign(taf + tbf)  # Etapa 2: elect sign
                ma = (_torch.sign(taf) == esign) | (taf == 0)  # Etapa 3
                mb = (_torch.sign(tbf) == esign) | (tbf == 0)
                cnt = (ma.float() + mb.float()).clamp(min=1)
                m = (taf * ma.float() + tbf * mb.float()) / cnt
                m[esign == 0] = (taf[esign == 0] + tbf[esign == 0]) / 2.0
                merged[key] = m.to(ta.dtype)
            _pathlib.Path(_output_dir).mkdir(parents=True, exist_ok=True)
            cfg_src = _pathlib.Path(dir_a) / 'adapter_config.json'
            if cfg_src.exists():
                _shutil.copy2(str(cfg_src), str(_pathlib.Path(_output_dir) / 'adapter_config.json'))
            if _use_sf:
                _save_sf(merged, str(_pathlib.Path(_output_dir) / 'adapter_model.safetensors'))
            else:
                _torch.save(merged, str(_pathlib.Path(_output_dir) / 'adapter_model.bin'))
            del ta_map, tb_map, merged
            _gc.collect()
            n_files = sum(1 for _ in _pathlib.Path(_output_dir).glob('*'))
            print(f'[TIES] Merge concluido | output={_output_dir} | files={n_files}')
            return _output_dir
        except Exception as _exc:
            print(f'[TIES][WARN] Merge falhou, usando adapter isolado: {_exc}')
            return None

    globals()['_apply_ties_merge'] = _apply_ties_merge
    print('[LoRA][COMPAT] Funcao TIES merge registrada.')


    if '_resolve_lora_adapter_dir' in globals() and '_orig_resolve_lora_adapter_dir_local_first' not in globals():
        _orig_resolve_lora_adapter_dir_local_first = _resolve_lora_adapter_dir

        def _resolve_lora_adapter_dir(cfg: CFG) -> str | None:
            """Resolve adapter local-first; aplica TIES merge se dois adapters disponiveis."""
            # Passo 1: encontrar melhor adapter local.
            _local_adapter = None
            candidates = _build_lora_resolution_candidates(cfg)
            for _, candidate in candidates:
                report = _inspect_adapter_dir(candidate)
                if report['valid']:
                    _local_adapter = candidate
                    break
            # Passo 2: tentar materializar adapter público HF.
            public_adapter_dir = _materialize_public_lora_adapter(cfg)
            _public_adapter = None
            if public_adapter_dir:
                public_report = _inspect_adapter_dir(public_adapter_dir)
                if public_report['valid']:
                    _public_adapter = public_adapter_dir
            # Passo 3: TIES merge quando ambos os adapters estao disponiveis.
            if _local_adapter and _public_adapter and _local_adapter != _public_adapter:
                _ties_fn = globals().get('_apply_ties_merge')
                if callable(_ties_fn):
                    try:
                        _merged = _ties_fn(_local_adapter, _public_adapter)
                        if _merged and _inspect_adapter_dir(_merged).get('valid', False):
                            print(f'[TIES] Adapter fundido ativo | output={_merged}')
                            return _merged
                    except Exception as _ties_exc:
                        print(f'[TIES][WARN] Merge falhou; usando adapter local: {_ties_exc}')
            # Fallback: local primeiro, depois publico.
            if _local_adapter:
                return _local_adapter
            if _public_adapter:
                return _public_adapter
            _print_lora_resolution_diagnostics(cfg, candidates)
            return None

        globals()['_resolve_lora_adapter_dir'] = _resolve_lora_adapter_dir
        print('[LoRA][COMPAT] Resolver atualizado: local-first + TIES merge (local+public HF).')

    globals()['_aimo3_lora_local_first_patch_applied'] = True
    print('[LoRA][COMPAT] Patch local-first aplicado com sucesso.')

[LoRA][COMPAT] Nenhum adapter local válido encontrado; mantendo fallback público.
[LoRA][COMPAT] Builder de candidatos ampliado para layouts de dataset local.
[LoRA][COMPAT] Funcao TIES merge registrada.
[LoRA][COMPAT] Resolver atualizado: local-first + TIES merge (local+public HF).
[LoRA][COMPAT] Patch local-first aplicado com sucesso.


In [19]:
# ===================== STABILITY PATCHES (FIRSTENV-ALIGNED + FAIL-FAST LORA) =====================
import contextlib
import gc
import os
import re


def _stability_env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}


# Defaults mais conservadores para aproximar o comportamento do firstenv.
CFG.use_adaptive_sampling = _stability_env_bool('AIMO3_USE_ADAPTIVE_SAMPLING', False)
CFG.reflexion_enabled = _stability_env_bool('AIMO3_REFLEXION_ENABLED', False)
CFG.verifier_enabled = _stability_env_bool('AIMO3_VERIFIER_ENABLED', False)
CFG.rerank_enabled = _stability_env_bool('AIMO3_RERANK_ENABLED', False)
CFG.tool_lite_enabled = _stability_env_bool('AIMO3_TOOL_LITE_ENABLED', False)
CFG.enable_loose_answer_extraction = _stability_env_bool('AIMO3_ENABLE_LOOSE_ANSWER_EXTRACTION', False)
CFG.strategy_hints_enabled = _stability_env_bool('AIMO3_ENABLE_STRATEGY_HINTS', False)
CFG.enable_ties_merge = _stability_env_bool('AIMO3_ENABLE_TIES_MERGE', False)
CFG.lora_require_adapter = _stability_env_bool('AIMO3_LORA_REQUIRE_ADAPTER', True)
CFG.lora_validate_with_peft = _stability_env_bool('AIMO3_LORA_VALIDATE_WITH_PEFT', False)
CFG.lora_peft_base_model_id = os.getenv(
    'AIMO3_LORA_PEFT_BASE_MODEL_ID',
    'unsloth/gpt-oss-120b-unsloth-bnb-4bit',
)


_ORIG_SCAN_FOR_ANSWER = globals().get('_orig_scan_for_answer')
_ORIG_SELECT_ANSWER = globals().get('_orig_select_answer')
_ORIG_PREPARE_PROBLEM_PROMPT = globals().get('_orig_prepare_problem_prompt')
_PATCHED_SELECT_ANSWER = globals().get('_patched_select_answer')
_PATCHED_PREPARE_PROBLEM_PROMPT = globals().get('_patched_prepare_problem_prompt')
_ORIGINAL_RESOLVE_LORA_ADAPTER_DIR = globals().get('_resolve_lora_adapter_dir')
_EXISTING_APPLY_TIES_MERGE = globals().get('_apply_ties_merge')
_INSPECT_ADAPTER_DIR = globals().get('_inspect_adapter_dir')


if not callable(_ORIG_SCAN_FOR_ANSWER):
    raise RuntimeError('[STABILITY] _orig_scan_for_answer não disponível para restaurar o parser estrito.')
if not callable(_ORIGINAL_RESOLVE_LORA_ADAPTER_DIR):
    raise RuntimeError('[STABILITY] _resolve_lora_adapter_dir não disponível para aplicar fail-fast do LoRA.')
if not callable(_INSPECT_ADAPTER_DIR):
    raise RuntimeError('[STABILITY] _inspect_adapter_dir não disponível para validar adapters.')


if callable(_EXISTING_APPLY_TIES_MERGE):
    def _gated_apply_ties_merge(
        dir_a: str,
        dir_b: str,
        output_dir: str | None = None,
        density: float = 0.20,
    ) -> str | None:
        if not getattr(CFG, 'enable_ties_merge', False):
            print('[TIES] Merge desativado por padrão; priorizando adapter local sem fusão.')
            return None
        return _EXISTING_APPLY_TIES_MERGE(dir_a, dir_b, output_dir=output_dir, density=density)

    globals()['_apply_ties_merge'] = _gated_apply_ties_merge


if _stability_env_bool('AIMO3_LORA_VALIDATE_WITH_PEFT', False):
    print('[LoRA][PEFT] Validação PEFT habilitada. O smoke test será executado quando o adapter for resolvido.')


def validate_lora_adapter_with_peft(adapter_dir: str, cfg: CFG) -> dict:
    """Executa um smoke test opcional do adapter usando Transformers + PEFT.

    O objetivo é validar rapidamente se o adapter pode ser carregado no formato PEFT.
    O teste é opt-in porque o modelo base de 120B em 4-bit ainda é pesado para o runtime.
    """
    if not getattr(cfg, 'lora_validate_with_peft', False):
        return {
            'validated': False,
            'skipped': True,
            'reason': 'validation_disabled',
            'adapter_dir': adapter_dir,
        }

    base_model = None
    model = None
    base_model_id = getattr(cfg, 'lora_peft_base_model_id', 'unsloth/gpt-oss-120b-unsloth-bnb-4bit')

    try:
        from peft import PeftModel
        from transformers import AutoModelForCausalLM

        print(f'[LoRA][PEFT] Validando adapter com base_model={base_model_id} | adapter_dir={adapter_dir}')
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            trust_remote_code=True,
            torch_dtype='auto',
            device_map='auto',
        )
        model = PeftModel.from_pretrained(base_model, adapter_dir)
        print('[LoRA][PEFT] Smoke test concluído com sucesso.')
        return {
            'validated': True,
            'skipped': False,
            'reason': 'ok',
            'adapter_dir': adapter_dir,
            'base_model_id': base_model_id,
        }
    finally:
        with contextlib.suppress(Exception):
            del model
        with contextlib.suppress(Exception):
            del base_model
        gc.collect()
        if 'torch' in globals():
            with contextlib.suppress(Exception):
                torch.cuda.empty_cache()


globals()['validate_lora_adapter_with_peft'] = validate_lora_adapter_with_peft


def _resolve_lora_adapter_dir_fail_fast(cfg: CFG) -> str | None:
    adapter_dir = _ORIGINAL_RESOLVE_LORA_ADAPTER_DIR(cfg)
    if adapter_dir is not None:
        report = _INSPECT_ADAPTER_DIR(adapter_dir)
        if not report.get('valid', False):
            raise RuntimeError(
                '[LoRA] Adapter resolvido, mas inválido. '
                f"path={adapter_dir} | reason={report.get('reason', 'unknown')}"
            )
        validate_lora_adapter_with_peft(adapter_dir, cfg)
        return adapter_dir

    if getattr(cfg, 'lora_enabled', False) and getattr(cfg, 'lora_require_adapter', True):
        explicit_path = (os.getenv('AIMO3_LORA_ADAPTER_PATH') or '').strip() or '<não definido>'
        public_repo = (os.getenv('AIMO3_LORA_HF_REPO_ID') or '').strip() or '<não definido>'
        raise RuntimeError(
            '[LoRA] LoRA está habilitado, mas nenhum adapter válido foi encontrado. '
            f'explicit_path={explicit_path} | hf_repo={public_repo}. '
            'Defina um adapter local válido ou habilite o download automático com artefatos acessíveis.'
        )

    return None


globals()['_resolve_lora_adapter_dir'] = _resolve_lora_adapter_dir_fail_fast


def _strict_scan_for_answer(self, text: str) -> int | None:
    value = _ORIG_SCAN_FOR_ANSWER(self, text)
    if value is not None:
        return value

    if getattr(self.cfg, 'enable_loose_answer_extraction', False):
        numeric_candidates = re.findall(r'(?<!\d)(\d{1,5})(?!\d)', text or '')
        for candidate in reversed(numeric_candidates):
            parsed = int(candidate)
            if 0 <= parsed <= 99999:
                return parsed

    return None


AIMO3Solver._scan_for_answer = _strict_scan_for_answer


if callable(_ORIG_PREPARE_PROBLEM_PROMPT) and callable(_PATCHED_PREPARE_PROBLEM_PROMPT):
    def _stable_prepare_problem_prompt(self, problem: str) -> tuple[str, dict]:
        if not getattr(self.cfg, 'strategy_hints_enabled', False):
            return _ORIG_PREPARE_PROBLEM_PROMPT(self, problem)
        return _PATCHED_PREPARE_PROBLEM_PROMPT(self, problem)

    AIMO3Solver._prepare_problem_prompt = _stable_prepare_problem_prompt


if callable(_ORIG_SELECT_ANSWER) and callable(_PATCHED_SELECT_ANSWER):
    def _stable_select_answer(self, detailed_results: list) -> int:
        if not any([
            getattr(self.cfg, 'rerank_enabled', False),
            getattr(self.cfg, 'verifier_enabled', False),
            getattr(self.cfg, 'reflexion_enabled', False),
        ]):
            return _ORIG_SELECT_ANSWER(self, detailed_results)
        return _PATCHED_SELECT_ANSWER(self, detailed_results)

    AIMO3Solver._select_answer = _stable_select_answer


print('[STABILITY] Patch aplicado: defaults seguros, parser estrito, LoRA fail-fast e seleção firstenv-like por padrão.')
print(
    '[STABILITY] Flags ativas => '
    f"adaptive_sampling={CFG.use_adaptive_sampling}, "
    f"verifier={CFG.verifier_enabled}, "
    f"reflexion={CFG.reflexion_enabled}, "
    f"rerank={CFG.rerank_enabled}, "
    f"tool_lite={CFG.tool_lite_enabled}, "
    f"loose_answer_extraction={CFG.enable_loose_answer_extraction}, "
    f"lora_require_adapter={CFG.lora_require_adapter}, "
    f"strategy_hints={CFG.strategy_hints_enabled}, "
    f"ties_merge={CFG.enable_ties_merge}"
)

[STABILITY] Patch aplicado: defaults seguros, parser estrito, LoRA fail-fast e seleção firstenv-like por padrão.
[STABILITY] Flags ativas => adaptive_sampling=False, verifier=False, reflexion=False, rerank=False, tool_lite=False, loose_answer_extraction=False, lora_require_adapter=True, strategy_hints=False, ties_merge=False


In [20]:
# Alinhado ao notebook base: inicialização direta do solver (sem fallback silencioso para 0).
print('[SOLVER] Inicializando AIMO3Solver com CFG...')
solver = AIMO3Solver(CFG)
print('[SOLVER] Inicialização concluída.')

[SOLVER] Inicializando AIMO3Solver com CFG...
Loading model weights from /kaggle/input/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 90.26 seconds.

[LoRA] Materializando adapter público: sibi-ai/gpt-oss-120b-aimo3-tir-lora


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/47.8M [00:00<?, ?B/s]

[LoRA] Adapter público pronto em: /kaggle/working/aimo3_downloaded_lora/gpt-oss-120b-aimo3-tir-lora
[SOLVER] LoRA ativo no vLLM: module=aimo3_lora | dir=/kaggle/working/aimo3_downloaded_lora/gpt-oss-120b-aimo3-tir-lora
[SOLVER] Modelo efetivo para geração: aimo3_lora
Waiting for vLLM server...
Server is ready (took 194.58 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 3.33 seconds.

[SOLVER] Inicialização concluída.


In [21]:
def predict(
    id_: pl.DataFrame,
    question: pl.DataFrame,
    answer: Optional[pl.DataFrame] = None,
) -> pl.DataFrame:
    """Resolve um ÃƒÆ’Ã‚Âºnico problema e devolve a resposta no formato esperado.

    Args:
        id_: DataFrame com o identificador do problema.
        question: DataFrame com o enunciado.
        answer: DataFrame opcional com a resposta esperada.

    Returns:
        DataFrame Polars com as colunas `id` e `answer`.
    """

    # Extrai os escalares do formato usado pelo gateway do Kaggle.
    id_value = id_.item(0)
    question_text = question.item(0)

    # Desabilita o GC durante a resoluÃƒÆ’Ã‚Â§ÃƒÆ’Ã‚Â£o para reduzir ruÃƒÆ’Ã‚Â­do de latÃƒÆ’Ã‚Âªncia.
    gc.disable()

    final_answer = solver.solve_problem(question_text)

    gc.enable()
    gc.collect()

    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [22]:
# ===================== INFERENCE SERVER COM MONITORAMENTO INTEGRADO =====================
# Este bloco substitui o `predict()` simples por uma versão com logging detalhado.
# O objetivo é facilitar auditoria de resultados sem alterar o contrato do gateway Kaggle.

import gc
import json
import os
import re
import time
from datetime import datetime, timezone
from typing import Optional

import pandas as pd
import polars as pl

if 'solver' not in globals():
    raise RuntimeError('[INFERENCE] `solver` não está definido. Execute a célula 12 antes da célula 14.')

try:
    import kaggle_evaluation.aimo_3_inference_server as aimo3_inference_server
except Exception as exc:
    raise RuntimeError('[INFERENCE] Módulo kaggle_evaluation indisponível no runtime.') from exc


class InferenceMonitor:
    """Monitora os problemas resolvidos durante a inferência."""

    def __init__(self, log_dir: str = 'inference_logs'):
        """Inicializa os arquivos de log e os contadores agregados."""

        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)

        self.problems_log = os.path.join(log_dir, 'problems_solved.jsonl')
        self.summary_log = os.path.join(log_dir, 'summary.csv')
        self.errors_log = os.path.join(log_dir, 'errors.log')

        self.results = []
        self.correct_count = 0
        self.wrong_count = 0
        self.error_count = 0

    def extract_problem_metadata(self, problem_text: str) -> list[str]:
        """Infere domínios matemáticos prováveis a partir do enunciado."""

        detector = globals().get('detect_problem_domains')
        if callable(detector):
            try:
                return detector(problem_text)
            except Exception:
                pass

        domains = []
        domain_keywords = {
            'geometry': r'triangle|circle|angle|perimeter|area|polygon|sphere|cube|tangent|square|rectangle',
            'number_theory': r'divisor|prime|modulo|mod|congruent|gcd|lcm|integer|coprime|remainder',
            'algebra': r'equation|polynomial|factor|expand|root|quadratic|function',
            'combinatorics': r'count|ways|permutation|combination|tournament|arrangement',
        }

        for domain, pattern in domain_keywords.items():
            if re.search(pattern, problem_text, re.IGNORECASE):
                domains.append(domain)

        return domains if domains else ['general']

    def verify_answer_basic(self, answer, problem_text: str) -> tuple[bool, str]:
        """Aplica verificações simples de sanidade sobre a resposta."""

        try:
            answer_int = int(answer)
        except Exception:
            return False, 'not_integer'

        if answer_int < 0:
            return False, 'negative_value'

        mod_match = re.search(r'mod\s+(\d+)|modulo\s+(\d+)', problem_text)
        if mod_match:
            modulus = int(mod_match.group(1) or mod_match.group(2))
            if not (0 <= answer_int < modulus):
                return False, f'exceeds_modulus_{modulus}'

        range_match = re.search(
            r'between\s+0\s+and\s+(\d+)|0.*?(\d+)',
            problem_text,
            re.IGNORECASE,
        )
        if range_match:
            upper = int(range_match.group(1) or range_match.group(2))
            if not (0 <= answer_int <= upper):
                return False, f'outside_range_0_{upper}'

        return True, 'sanity_check_passed'

    def log_problem(
        self,
        problem_id,
        problem_text: str,
        predicted_answer,
        execution_time: float,
        expected_answer=None,
        context_payload: Optional[dict] = None,
    ) -> dict:
        """Registra o resultado de um problema individual."""

        context_payload = context_payload or {}
        domains = context_payload.get('domains') or self.extract_problem_metadata(problem_text)
        is_reasonable, check_reason = self.verify_answer_basic(
            predicted_answer,
            problem_text,
        )

        theorem_titles = context_payload.get('theorem_titles', [])
        context_titles = context_payload.get('context_titles', [])
        context_source = context_payload.get('source', 'no_context')

        is_correct = None
        if expected_answer is not None:
            try:
                is_correct = int(predicted_answer) == int(expected_answer)
            except Exception:
                is_correct = False

        record = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'problem_id': problem_id,
            'problem_snippet': problem_text[:150].replace('\n', ' '),
            'predicted_answer': int(predicted_answer),
            'expected_answer': expected_answer,
            'is_correct': is_correct,
            'is_reasonable': is_reasonable,
            'sanity_check': check_reason,
            'domains': domains,
            'execution_time_seconds': execution_time,
            'context_source': context_source,
            'context_titles': context_titles,
            'context_items_count': len(context_payload.get('context_items', [])),
            'theorem_titles': theorem_titles,
            'theorem_count': len(context_payload.get('theorem_guidance', [])),
            'prompt_length': context_payload.get('prompt_length'),
        }

        self.results.append(record)

        if is_correct is True:
            self.correct_count += 1
            status = '✅ CORRETO'
        elif is_correct is False:
            self.wrong_count += 1
            status = '❌ ERRADO'
        else:
            status = '⚠️ NÃO_VERIFICADO'

        try:
            with open(self.problems_log, 'a', encoding='utf-8') as file_object:
                file_object.write(json.dumps(record, ensure_ascii=False) + '\n')
        except Exception as exc:
            self.error_count += 1
            self._log_error(f'Erro ao salvar JSONL: {exc}')

        print(f'\n{status} Problema {problem_id}')
        print(f"   Domínios: {', '.join(domains)}")
        print(f"   Fonte de contexto: {context_source}")
        print(f'   Resposta predita: {predicted_answer}')
        if expected_answer is not None:
            print(f'   Resposta esperada: {expected_answer}')
        print(f'   Tempo: {execution_time:.2f}s')
        if theorem_titles:
            print(f"   Teoremas/guias: {', '.join(theorem_titles[:3])}")
        if not is_reasonable:
            print(f'   ⚠️ Falha na verificação: {check_reason}')

        return record

    def _log_error(self, error_msg: str) -> None:
        """Registra erros internos do monitor sem interromper a inferência."""

        try:
            with open(self.errors_log, 'a', encoding='utf-8') as file_object:
                file_object.write(
                    f'{datetime.now(timezone.utc).isoformat()}: {error_msg}\n'
                )
        except Exception:
            pass

    def generate_summary(self) -> None:
        """Gera um resumo agregado ao final da execução local."""

        if not self.results:
            print('[WARN] Nenhum problema processado')
            return

        df = pd.DataFrame(self.results)
        df.to_csv(self.summary_log, index=False)

        total = len(self.results)
        correct = self.correct_count
        wrong = self.wrong_count
        accuracy = correct / max(1, correct + wrong)

        print('\n' + '=' * 70)
        print('📊 RESUMO FINAL DE INFERÊNCIA')
        print('=' * 70)
        print(f'Total de problemas: {total}')
        print(f'✅ Corretos: {correct}')
        print(f'❌ Errados: {wrong}')
        print(f'⚠️ Não verificados: {total - correct - wrong}')
        print(f'Acurácia: {accuracy:.1%}')

        if 'context_source' in df.columns:
            print('\nContexto por fonte:')
            for source, count in df['context_source'].fillna('no_context').value_counts().items():
                print(f'  - {source}: {count}')

        print(f'\nLogs salvos em: {self.log_dir}/')
        print('=' * 70)


inference_monitor = InferenceMonitor(log_dir='inference_logs')


def predict(
    id_: pl.DataFrame,
    question: pl.DataFrame,
    answer: Optional[pl.DataFrame] = None,
) -> pl.DataFrame:
    """Executa a inferência com monitoramento e tratamento defensivo de erro."""

    start_time = time.time()
    id_value = id_.item(0)
    question_text = question.item(0)
    expected_answer = answer.item(0) if answer is not None else None

    gc.disable()

    try:
        final_answer = solver.solve_problem(question_text)
    except Exception as exc:
        print(f'\n❌ ERRO ao resolver problema {id_value}: {exc}')
        inference_monitor._log_error(f'Problema {id_value}: {str(exc)[:100]}')
        final_answer = 0
    finally:
        gc.enable()
        gc.collect()

    execution_time = time.time() - start_time
    context_payload = getattr(solver, 'last_problem_context', None)

    inference_monitor.log_problem(
        problem_id=id_value,
        problem_text=question_text,
        predicted_answer=final_answer,
        execution_time=execution_time,
        expected_answer=expected_answer,
        context_payload=context_payload,
    )

    return pl.DataFrame({'id': id_value, 'answer': final_answer})


print('[OK] Monitoramento integrado ao predict()')
print()

inference_server = aimo3_inference_server.AIMO3InferenceServer(
    predict)

print('[INFO] Iniciando inference_server...')
print()

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('[MODO] Produção (competition rerun)')
    inference_server.serve()
else:
    print('[MODO] Teste local (local gateway)')
    reference_csv = os.getenv('AIMO3_REFERENCE_CSV') or globals().get('OFFLINE_PATHS', {}).get('reference_csv')
    if not reference_csv:
        raise RuntimeError('[INFERENCE] Caminho de reference.csv não resolvido. Defina AIMO3_REFERENCE_CSV.')
    inference_server.run_local_gateway((reference_csv,))
    print('\n[OK] Teste local concluído')
    inference_monitor.generate_summary()


[OK] Monitoramento integrado ao predict()

[INFO] Iniciando inference_server...

[MODO] Teste local (local gateway)

Problem: Let $\mathcal{F}$ be the set of functions $\alpha \colon \mathbb{Z}\to \mathbb{Z}$ for which there are only finitely many $n \in \mathbb{Z}$ such that $\alpha(n) \neq 0$. 

For two functions $\alpha$ and $\beta$ in $\mathcal{F}$, define their product $\alpha\star\beta$ to be $\sum\limits_{n\in\mathbb{Z}} \alpha(n)\cdot \beta(n)$. Also, for $n\in\mathbb{Z}$, define a shift operator $S_n \colon \mathcal{F}\to \mathcal{F}$ by $S_n(\alpha)(t)=\alpha(t+n)$ for all $t \in \mathbb{Z}$.

A function $\alpha \in \mathcal{F}$ is called \emph{shifty} if 
\begin{itemize}
    \item $\alpha(m)=0$ for all integers $m<0$ and $m>8$ and
    \item There exists $\beta \in \mathcal{F}$ and integers $k \neq l$ such that for all $n \in \mathbb{Z}$
    \begin{equation*}
        S_n(\alpha)\star\beta =
        \begin{cases}
            1 & n \in \{k,l\} \\
            0 & n \not \in \{k,

,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,7,11401,0,0,0.802,34
1,9,12564,2,0,0.814,266
2,1,14459,7,0,0.686,80
3,2,15195,12,2,0.779,266
4,10,16439,15,0,0.797,160
5,6,16763,8,0,0.804,160
6,4,17602,10,1,0.813,160
7,8,18690,8,0,0.758,44
8,5,30714,0,0,0.704,126
9,3,30810,44,2,0.654,160


,Answer,Votes,Score
0,160,4,5.258
1,266,2,2.513
2,80,1,1.459
3,126,1,1.421
4,44,1,1.320
5,34,1,1.247



Final Answer: 160


✅ CORRETO Problema dd7f5e
   Domínios: number_theory, algebra
   Fonte de contexto: compat_plain_prompt
   Resposta predita: 160
   Resposta esperada: 160
   Tempo: 409.94s
   Teoremas/guias: Congruence classes and constructive checking
   ⚠️ Falha na verificação: outside_range_0_0

Problem: Alice and Bob are each holding some integer number of sweets. Alice says to Bob: ``If we each added the number of sweets we're holding to our (positive integer) age, my answer would be double yours. If we took the product, then my answer would be four times yours.'' Bob replies: ``Why don't you give me five of your sweets because then both our sum and product would be equal.'' What is the product of Alice and Bob's ages?

Context: source=compat_plain_prompt | domains=number_theory | snippets=0 | theorems=1
Budget: 900.00 seconds | Deadline: 1773630862.39



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,3,1579,1,0,0.550,50
1,1,2054,1,0,0.658,50
2,9,2074,1,0,0.574,50
3,10,2821,1,0,0.507,50
4,6,3063,0,0,0.428,50
5,8,3223,1,0,0.651,50
6,4,3225,2,0,0.586,50


...

,Answer,Votes,Score
0,32951,7,11.944



Final Answer: 32951


✅ CORRETO Problema 26de63
   Domínios: number_theory, algebra
   Fonte de contexto: compat_plain_prompt
   Resposta predita: 32951
   Resposta esperada: 32951
   Tempo: 86.86s
   Teoremas/guias: Euler theorem and totient workflow, Congruence classes and constructive checking, Modular exponentiation, order, and CRT workflow
   ⚠️ Falha na verificação: outside_range_0_24

Problem: A tournament is held with $2^{20}$ runners each of which has a different running speed. In each race, two runners compete against each other with the faster runner always winning the race. The competition consists of $20$ rounds with each runner starting with a score of $0$. In each round, the runners are paired in such a way that in each pair, both runners have the same score at the beginning of the round. The winner of each race in the $i^{\text{th}}$ round receives $2^{20-i}$ points and the loser gets no points.

At the end of the tournament, we rank the competitors according to their 

,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,6,5889,1,0,1.033,62140
1,2,8520,3,0,1.031,62140
2,4,15486,20,3,0.849,21818
3,8,16396,0,0,0.836,24287
4,3,16446,12,1,0.833,21818
5,7,18287,22,3,0.755,21818
6,9,21075,17,1,0.908,62140
7,10,23751,8,0,0.769,62140
8,5,26468,26,3,0.666,21818
9,1,29007,24,2,0.828,21818


,Answer,Votes,Score
0,21818,5,6.412
1,62140,4,4.340
2,24287,1,1.196



Final Answer: 21818


✅ CORRETO Problema 424e18
   Domínios: number_theory, combinatorics
   Fonte de contexto: compat_plain_prompt
   Resposta predita: 21818
   Resposta esperada: 21818
   Tempo: 371.34s
   Teoremas/guias: Euler theorem and totient workflow, Congruence classes and constructive checking, Modular exponentiation, order, and CRT workflow
   ⚠️ Falha na verificação: outside_range_0_20

[OK] Teste local concluído

📊 RESUMO FINAL DE INFERÊNCIA
Total de problemas: 10
✅ Corretos: 9
❌ Errados: 1
⚠️ Não verificados: 0
Acurácia: 90.0%

Contexto por fonte:
  - compat_plain_prompt: 10

Logs salvos em: inference_logs/
